<a href="https://colab.research.google.com/github/enicaking/tfg-vae/blob/main/6_dscvi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Discrete Error‑Correcting Uncoded VAE for scRNA‑seq (codedVAE + scVI)

This notebook replicates the core idea of the codedVAE paper (OpenReview `66P2ioAYY4`) in a **scVI** framework for single‑cell transcriptomics.

1. Load a toy heart atlas dataset (scvi‑tools).
2. Train a standard continuous‑latent scVI model.
3. Implement a discrete‑latent model that uses the ECC coded scheme
4. Compare latent spaces, reconstruction quality, and verify the correctness of the coded scheme.
5. Gridsearch on objective function KL weight and Coded DSCVI learning process.


# 0. Environment

## Libraries

In [ ]:
import scvi

In [ ]:
import torch
print(torch.cuda.is_available())      # True
print(torch.cuda.get_device_name(0))  # Tesla T4
print(f"PyTorch version: {torch.__version__}")

In [ ]:
from __future__ import annotations

import os
import math
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()
# import umap # gives error so use scanpy.pl.umap

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import ModuleList
from torch.nn.functional import one_hot
from torch.distributions import Normal, RelaxedBernoulli, Bernoulli, kl_divergence
torch.set_float32_matmul_precision("high") # investigate what this helps to do


import logging
import warnings
from typing import TYPE_CHECKING

import collections
if TYPE_CHECKING:
    from collections.abc import Callable, Iterable
    from typing import Literal
    from torch.distributions import Distribution

logger = logging.getLogger(__name__)

import tempfile

In [ ]:
import scvi
from scvi import REGISTRY_KEYS, settings
from scvi.model import SCVI
from scvi.model._utils import get_max_epochs_heuristic
from scvi.module import VAE
from scvi.model.base import (
    BaseModelClass,
    VAEMixin,
    RNASeqMixin,
    ArchesMixin,
    UnsupervisedTrainingMixin,
    BaseMinifiedModeModelClass,
)
from scvi.dataloaders import DataSplitter
from scvi.train import TrainingPlan, TrainRunner

from scvi.model._utils import _init_library_size
from scvi.model.base import UnsupervisedTrainingMixin
from scvi.model.utils import get_minified_adata_scrna

from scvi.data import AnnDataManager
from scvi.data.fields import (
    LayerField,
    NumericalObsField,
    CategoricalObsField,
    CategoricalJointObsField,
    NumericalJointObsField
)

from scvi.data._constants import ADATA_MINIFY_TYPE
from scvi.module._constants import MODULE_KEYS
from scvi.module.base import (
    BaseMinifiedModeModuleClass,
    EmbeddingModuleMixin,
    LossOutput,
    auto_move_data,
)
from scvi.utils import setup_anndata_dsp
from scvi.utils import unsupported_if_adata_minified
from scvi.nn._utils import ExpActivation

scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)


## Directory

In [ ]:
# Working locally
from pathlib import Path
import os
from datetime import datetime

project_root = Path.cwd()

SAVE_DIR = project_root / "scvi_data"
NOTEBOOK_DIR = project_root
PLOTS_DIR = SAVE_DIR / "plots"
TABLES_DIR = SAVE_DIR / "tables"
MODELS_DIR = SAVE_DIR / "models"
DEFAULT_MODEL_NAME = "model_v1"

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(NOTEBOOK_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)


## Helper functions
1. `dclamp` adds numerical stability for reparametrization after sigmoid transformation
2. [`sample_from_qz_given_x`](https://github.com/mariamartinezgarcia/codedVAE/blob/main/src/utils/sampling.py) implements the reparametrization to allow posterior/encoder q(m,z|x) to be calculated using the marginal q(z|x) = q(m|x)p(z|m). q(m|x) is the encoder and p(z|m) is the smoothing transformation.
3. `kl_div_bernoulli` implementation (based on [equation](https://statproofbook.github.io/P/bern-kl.html))
4. `build_repetition_G(bits_m, r)`
5. `modulate_words` - figuring that one out
6. `plot_path(plot_type: str, model_name: str = "default_model")`
7. `model_path(model_name: str)` 

In [ ]:
import torch
import torch.distributions as dist
from torch.amp import custom_bwd, custom_fwd

class DifferentiableClamp(torch.autograd.Function):
    @staticmethod
    @custom_fwd(device_type="cuda")
    def forward(ctx, input, min, max):
        return input.clamp(min=min, max=max)

    @staticmethod
    @custom_bwd(device_type="cuda")
    def backward(ctx, grad_output):
        return grad_output.clone(), None, None

def dclamp(input, min, max):
    return DifferentiableClamp.apply(input, min, max)

def sample_from_qz_given_x(qi, beta=torch.tensor(10), n_samples=1):
    assert torch.all(qi >= 0), "Negative value encountered in bit probabilities."
    assert torch.all(qi <= 1), "Value larger than 1 encountered in bit probabilities."
    epsilon = 1e-6
    qi = qi.unsqueeze(2).repeat(1, 1, n_samples)
    ones = torch.ones_like(qi)
    qi = dclamp(qi, 1e-3, 1 - 1e-3)
    rho = torch.rand_like(qi)
    b = (rho + torch.exp(-beta) * (qi - rho)) / (ones - qi) - ones
    c = -(qi * torch.exp(-beta)) / (ones - qi)
    dif = torch.sqrt(torch.pow(b, 2) - 4 * c) - b
    dif = torch.where(dif <= 0, epsilon, dif)
    q_z = (-1 / beta) * torch.log(dif / 2)
    assert torch.all(torch.isfinite(q_z)), "Invalid q(z|x) value (inf or nan)."
    return q_z

def kl_div_bernoulli(q_probs, p_probs):
    if isinstance(q_probs, dist.Bernoulli):
        q_probs = q_probs.probs
    if isinstance(p_probs, dist.Bernoulli):
        p_probs = p_probs.probs
    q_probs = torch.clamp(q_probs, min=1e-6, max=1 - 1e-6)
    p_probs = torch.clamp(p_probs, min=1e-6, max=1 - 1e-6)
    kl_div = q_probs * (torch.log(q_probs) - torch.log(p_probs))
    kl_div += (1 - q_probs) * (torch.log(1 - q_probs) - torch.log(1 - p_probs))
    return torch.sum(kl_div, dim=1)

def build_repetition_G(bits_m, r):
    num_cols = bits_m * r
    repetitions = num_cols // bits_m
    generator = torch.zeros((bits_m, num_cols))
    for i in range(bits_m):
        generator[i, i * repetitions:(i + 1) * repetitions] = 1
    return generator

In [ ]:
def plot_path(plot_type: str,
              model_name: str = "default_model",
              ext: str = "png") -> str:
    """
    Build a consistent, collision-free path for a plot file.

    e.g. plot_path("umap")                 -> plots/default_model_umap.png
         plot_path("tsne", "my_model_v2")  -> plots/my_model_v2_tsne.png

    If a file with that exact name already exists, a numeric suffix is
    appended automatically (_1, _2, ...) so you never silently overwrite
    an earlier model's plot -- even if you forget to change model_name.
    """
    # Ensure the plots directory exists
    # PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    
    # Build the base filename
    fname = f"{model_name}_{plot_type}"

    # Check for collisions and add suffix if needed
    candidate = PLOTS_DIR / f"{fname}.{ext}"
    counter = 1
    while candidate.exists():
        candidate = PLOTS_DIR / f"{fname}_{counter}.{ext}"
        counter += 1
    
    return str(candidate)


In [ ]:
def model_path(model_name) -> str:
    """Consistent checkpoint folder for `model.save(...)` / `Model.load(...)`."""
    model_path =  project_root / "scvi_data" / "models" / model_name
    return model_path

In [ ]:
REGISTERED_MODELS: set[str] = set()

def register_model(model_name: str) -> str:
    """
    Call this once per new model/experiment, right when you name it.

    Doesn't do anything magic -- it just validates the name, remembers it,
    and returns it back so you can do:
        MODEL_NAME = register_model("uncoded_dscvi_v2")
    and use MODEL_NAME everywhere below instead of retyping the string.
    """
    clean_name = model_name.strip().replace(" ", "_")
    if clean_name in REGISTERED_MODELS:
        print(f"[register_model] Warning: '{clean_name}' was already "
              f"registered this session. Re-using it (existing plots for "
              f"this name will get an auto-incremented suffix, not "
              f"overwritten).")
    REGISTERED_MODELS.add(clean_name)
    return clean_name

def list_registered_models() -> list[str]:
    return sorted(REGISTERED_MODELS)


In [ ]:
# from src.nn.modules import dclamp
# This is the function that takes m -> G -> z (i.e. p(z|c))

def modulate_words(words, beta=10):

    """
    Modulate words using the smoothing transformations.

    Parameters
        ----------
        words: torch.tensor
            Words to be modulated.
        beta : float
            Temperature term that controls the decay of the exponentials in the smoothing transformation. Default to 10.

        Returns
        -------
        z: torch.tensor
            Modulated words.
    """

    # Here we are modulating C using the exponentials
    # Obtain sample from p(z|c) in the coded case -> we fix the code word (it was previously sampled from q(c|x,C))
    # c_sample has shape [N, n_bits, n_samples] -> we obtain a sample from p(z|c) for each sampled word c

    # Sanity check
    assert torch.all(torch.logical_or(words == 0, words == 1)), "Invalid word encountered. All words should be binary vectors."

    epsilon = 1e-6

    # Create masks with the sampled word to select the bit samples
    not_c_sample = torch.logical_not(words).type(torch.float32) # shape [N, n_bits, n_samples], ones where ci=0

    # Auxiliar variable
    ones = torch.ones((words.shape)).to(words.device)

    # Sample from U(0,1)
    rho = torch.rand(words.shape).to(words.device)

    # Sample from F^(-1)(zi|ci=0)
    z0 = -(1/beta)*torch.log((ones-rho*(ones-ones*torch.exp(-beta)))+epsilon)
    # Apply ci=0 mask
    z0 = z0*not_c_sample

    # Sample from F^(-1)(zi|ci=1)
    z1 = (1/beta)*torch.log((rho*(ones-ones*torch.exp(-beta) + ones*torch.exp(-beta)))+epsilon) + 1
    # Apply ci=1 mask
    z1 = z1*words

    z = z0+z1

    # Sanity check
    assert torch.any(torch.isinf(z))==False, "Invalid z_sample value (inf)."
    assert torch.any(torch.isnan(z))==False, "Invalid z_sample value (nan)."

    return z  # shape [N, n_bits, n_samples]

# 1. Data Preprocessing

In [ ]:
# Load heart cell atlas subsampled
adata = scvi.data.heart_cell_atlas_subsampled()
print(adata)
import scanpy as sc
# Basic filtering and HVG selection
sc.pp.filter_genes(adata, min_counts=3) # Minimum gene expressed in 3 cells
adata.layers["counts"] = adata.X.copy()
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat_v3")
adata = adata[:, adata.var.highly_variable].copy()
adata.shape

# 2. Discrete scVI (DSCVI)

Changes made
- Added `self.beta` temperature term at default value 10. Beta is the inverse temperature term of Gumbel-Softmax smoothing. The higher it is, the stricter (values go to 0 or 1 more easily)
- Added `'ber'` as an option to `self.latent_distribution` for uncoded DVAE++

Important functions from `coded_vae.py`
- [forward()](https://github.com/mariamartinezgarcia/codedVAE/blob/main/src/coded_vae.py#L127)
- [generate()](https://github.com/mariamartinezgarcia/codedVAE/blob/main/src/coded_vae.py#L324)

Functions native to scVI
- get_latent_representation() in UncodedSCVI()
- loss() in UncodedSCVIModule()
- sample() in UncodedSCVIModule()
- generative() in UncodedSCVIModule()
- _regular_inference() in UncodedSCVIModule()


NN architecture links
- [Encoder](https://github.com/mariamartinezgarcia/codedVAE/blob/main/src/nn/encoder.py) (dcgan)
- [Decoder](https://github.com/mariamartinezgarcia/codedVAE/blob/main/src/nn/decoder.py) (cnnskip)

In [ ]:
# Encoder
import collections
from collections.abc import Callable, Iterable
from typing import Literal

import torch
from torch import nn
from torch.distributions import Normal
from torch.distributions import Normal, RelaxedBernoulli, Bernoulli, kl_divergence
from torch.nn import ModuleList

from scvi.nn._utils import ExpActivation


def _identity(x):
    return x


class FCLayers(nn.Module):
    """A helper class to build fully-connected layers for a neural network.

    Parameters
    ----------
    n_in
        The dimensionality of the input
    n_out
        The dimensionality of the output
    n_cat_list
        A list containing, for each category of interest,
        the number of categories. Each category will be
        included using a one-hot encoding.
    n_cont
        The dimensionality of the continuous covariates
    n_layers
        The number of fully-connected hidden layers
    n_hidden
        The number of nodes per hidden layer
    dropout_rate
        Dropout rate to apply to each of the hidden layers
    use_batch_norm
        Whether to have `BatchNorm` layers or not
    use_layer_norm
        Whether to have `LayerNorm` layers or not
    use_activation
        Whether to have layer activation or not
    bias
        Whether to learn bias in linear layers or not
    inject_covariates
        Whether to inject covariates in each layer, or just the first (default).
    activation_fn
        Which activation function to use
    """

    def __init__(
        self,
        n_in: int,
        n_out: int,
        n_cat_list: Iterable[int] = None,
        n_cont: int = 0,
        n_layers: int = 1,
        n_hidden: int = 128,
        dropout_rate: float = 0.1,
        use_batch_norm: bool = True,
        use_layer_norm: bool = False,
        use_activation: bool = True,
        bias: bool = True,
        inject_covariates: bool = True,
        activation_fn: nn.Module = nn.ReLU,
    ):
        super().__init__()
        self.inject_covariates = inject_covariates
        layers_dim = [n_in] + (n_layers - 1) * [n_hidden] + [n_out]

        if n_cat_list is not None:
            # n_cat = 1 will be ignored
            self.n_cat_list = [n_cat if n_cat > 1 else 0 for n_cat in n_cat_list]
        else:
            self.n_cat_list = []

        self.n_cov = n_cont + sum(self.n_cat_list)

        self.fc_layers = nn.Sequential(
            collections.OrderedDict(
                [
                    (
                        f"Layer {i}",
                        nn.Sequential(
                            nn.Linear(
                                n_in + self.n_cov * self.inject_into_layer(i),
                                n_out,
                                bias=bias,
                            ),
                            # non-default params come from defaults in the original Tensorflow
                            # implementation
                            nn.BatchNorm1d(n_out, momentum=0.01, eps=0.001)
                            if use_batch_norm
                            else None,
                            nn.LayerNorm(n_out, elementwise_affine=False)
                            if use_layer_norm
                            else None,
                            activation_fn() if use_activation else None,
                            nn.Dropout(p=dropout_rate) if dropout_rate > 0 else None,
                        ),
                    )
                    for i, (n_in, n_out) in enumerate(
                        zip(layers_dim[:-1], layers_dim[1:], strict=True)
                    )
                ]
            )
        )

    def inject_into_layer(self, layer_num) -> bool:
        """Helper to determine if covariates should be injected."""
        user_cond = layer_num == 0 or (layer_num > 0 and self.inject_covariates)
        return user_cond

    def set_online_update_hooks(self, hook_first_layer=True):
        """Set online update hooks."""
        self.hooks = []

        def _hook_fn_weight(grad):
            categorical_dims = sum(self.n_cat_list)
            new_grad = torch.zeros_like(grad)
            if categorical_dims > 0:
                new_grad[:, -categorical_dims:] = grad[:, -categorical_dims:]
            return new_grad

        def _hook_fn_zero_out(grad):
            return grad * 0

        for i, layers in enumerate(self.fc_layers):
            for layer in layers:
                if i == 0 and not hook_first_layer:
                    continue
                if isinstance(layer, nn.Linear):
                    if self.inject_into_layer(i):
                        w = layer.weight.register_hook(_hook_fn_weight)
                    else:
                        w = layer.weight.register_hook(_hook_fn_zero_out)
                    self.hooks.append(w)
                    b = layer.bias.register_hook(_hook_fn_zero_out)
                    self.hooks.append(b)

    def forward(self, x: torch.Tensor, *cat_list: int, cont: torch.Tensor | None = None):
        """Forward computation on ``x``.

        Parameters
        ----------
        x
            tensor of values with shape ``(n_in,)``
        cat_list
            list of category membership(s) for this sample
        cont
            tensor of continuous covariates with shape ``(n_cont,)``

        Returns
        -------
        :class:`torch.Tensor`
            tensor of shape ``(n_out,)``
        """
        one_hot_cat_list = []  # for generality in this list many idxs useless.
        cont_list = [cont] if cont is not None else []
        cat_list = cat_list or []

        if len(self.n_cat_list) > len(cat_list):
            raise ValueError("nb. categorical args provided doesn't match init. params.")
        for n_cat, cat in zip(self.n_cat_list, cat_list, strict=False):
            if n_cat and cat is None:
                raise ValueError("cat not provided while n_cat != 0 in init. params.")
            if n_cat > 1:  # n_cat = 1 will be ignored - no additional information
                if cat.size(1) != n_cat:
                    one_hot_cat = nn.functional.one_hot(cat.squeeze(-1), n_cat)
                else:
                    one_hot_cat = cat  # cat has already been one_hot encoded
                one_hot_cat_list += [one_hot_cat]
        cov_list = cont_list + one_hot_cat_list
        for i, layers in enumerate(self.fc_layers):
            for layer in layers:
                if layer is not None:
                    if isinstance(layer, nn.BatchNorm1d):
                        if x.dim() == 3:
                            if (
                                x.device.type == "mps"
                            ):  # TODO: remove this when MPS supports for loop.
                                x = torch.cat(
                                    [(layer(slice_x.clone())).unsqueeze(0) for slice_x in x], dim=0
                                )
                            else:
                                x = torch.cat(
                                    [layer(slice_x).unsqueeze(0) for slice_x in x], dim=0
                                )
                        else:
                            x = layer(x)
                    else:
                        if isinstance(layer, nn.Linear) and self.inject_into_layer(i):
                            if x.dim() == 3:
                                cov_list_layer = [
                                    o.unsqueeze(0).expand((x.size(0), o.size(0), o.size(1)))
                                    for o in cov_list
                                ]
                            else:
                                cov_list_layer = cov_list
                            x = torch.cat((x, *cov_list_layer), dim=-1)
                        x = layer(x)
        return x


import torch
import torch.nn as nn
from torch.distributions import Bernoulli, kl_divergence

def _identity(x):
    return x

class Encoder(nn.Module):
    """
    Encode data into a latent space of n_output dimensions.

    Supports:
    - "normal" : Gaussian (mean, variance)
    - "ln"     : Log‑Normal (softmaxed)
    - "ber"    : Bernoulli, using custom reparameterization trick (inverse cdf beta smoothing)
    """

    def __init__(
        self,
        n_input: int,
        n_output: int,
        n_cat_list: Iterable[int] = None,
        n_layers: int = 1,
        n_hidden: int = 128,
        dropout_rate: float = 0.1,
        distribution: str = "ber",
        coded_key: bool = False,
        G = None,
        var_eps: float = 1e-4,
        var_activation: Callable | None = None,
        return_dist: bool = False,
        beta: float = 10.0, # reparameterization temperature parameter
        **kwargs,
    ):
        super().__init__()

        self.distribution = distribution
        self.coded_key = coded_key
        self.var_eps = var_eps
        self.return_dist = return_dist
        self.beta = beta
        if G is not None:
          self.register_buffer("G", G)
          self.register_buffer("H", G.T.contiguous()) # why contiguous?
        else:
          self.G=None
          self.H=None
        # self.G = G
        # if G is not None:
        #   self.H = G.T
        # self.device = 'cpu' # change in the future? placeholder

        # Shared encoder
        self.encoder = FCLayers(
            n_in=n_input,
            n_out=n_hidden,
            n_cat_list=n_cat_list,
            n_layers=n_layers,
            n_hidden=n_hidden,
            dropout_rate=dropout_rate,
            **kwargs,
        )

        # Mean / logits head
        self.mean_encoder = nn.Linear(n_hidden, n_output)

        # Variance head only for continuous distributions
        if distribution in ["normal", "ln"]:
            self.var_encoder = nn.Linear(n_hidden, n_output)
            self.var_activation = torch.exp if var_activation is None else var_activation
        else: # Bernoulli distribution output does not have variance metric
            self.var_encoder = None
            self.var_activation = None

        # Transformations for non‑Normal outputs (only used for "ln")
        if distribution == "ln":
            self.z_transformation = nn.Softmax(dim=-1)
        else:
            self.z_transformation = _identity

    def forward(self, x: torch.Tensor, *cat_list: int):
        """
        Forward pass.

        For distribution="ber":
            - Computes logits from mean_encoder.
            - Converts to probabilities (qi) via sigmoid.
            - Uses your sample_from_qz_given_x to obtain a continuous sample (reparameterized).
            - If return_dist=True, returns (Bernoulli distribution, latent_sample).
            - Else returns (logits, probs, latent_sample).
        """
        q = self.encoder(x, *cat_list)

        if self.distribution == "ber":
            logits = self.mean_encoder(q)                     # (batch, n_latent) or (batch, n_code)
            qi = torch.sigmoid(logits)                        # probabilities in (0,1)
            qi = dclamp(qi, 1e-4, 1-1e-4)
            if torch.isnan(logits).any() or torch.isinf(logits).any():
              print(f"Bad logits detected: min={logits.min()}, max={logits.max()}")

            if not self.coded_key: # UNCODED CASE
              # Use your reparameterization function to get a sample
              # (we call it with n_samples=1 to get a single sample per batch element)
              latent = sample_from_qz_given_x(qi, beta=torch.tensor(self.beta), n_samples=1)
              latent = latent.squeeze(-1) # (batch, n_latent)
              q_m = qi                    # n_latent -dim, same tensor, has gradients
              return qi, q_m, latent

            else: # CODED CASE
              # Use your reparameterization function to get a sample
              # (we call it with n_samples=1 to get a single sample per batch element)

              # ENCODER_OUT IS p(c|x)
              # Soft decoding: obtain the marginals of the information bits

              logpm1 = torch.matmul(torch.log(qi), self.H)
              logpm0 = torch.matmul(torch.log(1-qi), self.H)

              log_marginals = torch.stack((logpm0, logpm1), dim=2) # THIS IS p(m|x)
              log_marginals_norm = log_marginals - torch.logsumexp(log_marginals, dim=-1, keepdim=True)
              p_m_given_x = torch.exp(log_marginals_norm[:,:,1]) # (batch, n_latent)

              # Introduce code structure to get p(z|x)
              qc = torch.matmul(p_m_given_x, self.G)
              qc = dclamp(qc, 1e-4, 1 - 1e-4) # Numerical stability

              # Modulate c to obtain z
              latent = sample_from_qz_given_x(qc, beta=torch.tensor(self.beta), n_samples=1)
              latent = latent.squeeze(-1)          # (batch, n_latent, 1)  --> (batch, n_latent)

              return qc, p_m_given_x, latent

            # # Create a Bernoulli distribution for KL divergence (optional)
            # dist = Bernoulli(probs=qi)
            # # Should maybe rethink this, return qi and empty variable no matter what ?

            # if self.return_dist:
            #     return dist, latent
            # else:
            #     return qi, latent

        else:
            # Normal or Log‑Normal (unchanged)
            q_m = self.mean_encoder(q)
            q_v = self.var_activation(self.var_encoder(q)) + self.var_eps
            dist = Normal(q_m, q_v.sqrt())
            latent = self.z_transformation(dist.rsample())

            if self.return_dist:
                return dist, latent
            else:
                return q_m, q_v, latent


class DSCVIModule(EmbeddingModuleMixin, BaseMinifiedModeModuleClass):
    """Variational auto-encoder :cite:p:`Lopez18`.

    Parameters
    ----------
    n_input
        Number of input features.
    n_batch
        Number of batches. If ``0``, no batch correction is performed.
    n_labels
        Number of labels.
    n_hidden
        Number of nodes per hidden layer. Passed into :class:`~scvi.nn.Encoder` and
        :class:`~scvi.nn.DecoderSCVI`.
    n_latent
        Dimensionality of the latent space.
    n_layers
        Number of hidden layers. Passed into :class:`~scvi.nn.Encoder` and
        :class:`~scvi.nn.DecoderSCVI`.
    n_continuous_cov
        Number of continuous covariates.
    n_cats_per_cov
        A list of integers containing the number of categories for each categorical covariate.
    dropout_rate
        Dropout rate. Passed into :class:`~scvi.nn.Encoder` but not :class:`~scvi.nn.DecoderSCVI`.
    dispersion
        Flexibility of the dispersion parameter when ``gene_likelihood`` is either ``"nb"`` or
        ``"zinb"``. One of the following:

        * ``"gene"``: parameter is constant per gene across cells.
        * ``"gene-batch"``: parameter is constant per gene per batch.
        * ``"gene-label"``: parameter is constant per gene per label.
        * ``"gene-cell"``: parameter is constant per gene per cell.
    log_variational
        If ``True``, use :func:`~torch.log1p` on input data before encoding for numerical stability
        (not normalization).
    gene_likelihood
        Distribution to use for reconstruction in the generative process. One of the following:

        * ``"nb"``: :class:`~scvi.distributions.NegativeBinomial`.
        * ``"zinb"``: :class:`~scvi.distributions.ZeroInflatedNegativeBinomial`.
        * ``"poisson"``: :class:`~scvi.distributions.Poisson`.
        * ``"normal"``: :class:`~torch.distributions.Normal`.
    latent_distribution
        Distribution to use for the latent space. One of the following:

        * ``"normal"``: isotropic normal.
        * ``"ln"``: logistic normal with normal params N(0, 1).
    encode_covariates
        If ``True``, covariates are concatenated to gene expression prior to passing through
        the encoder(s). Else, only gene expression is used.
    deeply_inject_covariates
        If ``True`` and ``n_layers > 1``, covariates are concatenated to the outputs of hidden
        layers in the encoder(s) (if ``encoder_covariates`` is ``True``) and the decoder prior to
        passing through the next layer.
    batch_representation
        ``EXPERIMENTAL`` Method for encoding batch information. One of the following:

        * ``"one-hot"``: represent batches with one-hot encodings.
        * ``"embedding"``: represent batches with continuously-valued embeddings using
          :class:`~scvi.nn.Embedding`.

        Note that batch representations are only passed into the encoder(s) if
        ``encode_covariates`` is ``True``.
    use_batch_norm
        Specifies where to use :class:`~torch.nn.BatchNorm1d` in the model. One of the following:

        * ``"none"``: don't use batch norm in either encoder(s) or decoder.
        * ``"encoder"``: use batch norm only in the encoder(s).
        * ``"decoder"``: use batch norm only in the decoder.
        * ``"both"``: use batch norm in both encoder(s) and decoder.

        Note: if ``use_layer_norm`` is also specified, both will be applied (first
        :class:`~torch.nn.BatchNorm1d`, then :class:`~torch.nn.LayerNorm`).
    use_layer_norm
        Specifies where to use :class:`~torch.nn.LayerNorm` in the model. One of the following:

        * ``"none"``: don't use layer norm in either encoder(s) or decoder.
        * ``"encoder"``: use layer norm only in the encoder(s).
        * ``"decoder"``: use layer norm only in the decoder.
        * ``"both"``: use layer norm in both encoder(s) and decoder.

        Note: if ``use_batch_norm`` is also specified, both will be applied (first
        :class:`~torch.nn.BatchNorm1d`, then :class:`~torch.nn.LayerNorm`).
    use_size_factor_key
        If ``True``, use the :attr:`~anndata.AnnData.obs` column as defined by the
        ``size_factor_key`` parameter in the model's ``setup_anndata`` method as the scaling
        factor in the mean of the conditional distribution. Takes priority over
        ``use_observed_lib_size``.
    use_observed_lib_size
        If ``True``, use the observed library size for RNA as the scaling factor in the mean of the
        conditional distribution.
    extra_payload_autotune
        If ``True``, will return extra matrices in the loss output to be used during autotune
    library_log_means
        :class:`~numpy.ndarray` of shape ``(1, n_batch)`` of means of the log library sizes that
        parameterize the prior on library size if ``use_size_factor_key`` is ``False`` and
        ``use_observed_lib_size`` is ``False``.
    library_log_vars
        :class:`~numpy.ndarray` of shape ``(1, n_batch)`` of variances of the log library sizes
        that parameterize the prior on library size if ``use_size_factor_key`` is ``False`` and
        ``use_observed_lib_size`` is ``False``.
    var_activation
        Callable used to ensure positivity of the variance of the variational distribution. Passed
        into :class:`~scvi.nn.Encoder`. Defaults to :func:`~torch.exp`.
    extra_encoder_kwargs
        Additional keyword arguments passed into :class:`~scvi.nn.Encoder`.
    extra_decoder_kwargs
        Additional keyword arguments passed into :class:`~scvi.nn.DecoderSCVI`.
    batch_embedding_kwargs
        Keyword arguments passed into :class:`~scvi.nn.Embedding` if ``batch_representation`` is
        set to ``"embedding"``.

    """

    def __init__(
        self,
        n_input: int,
        n_batch: int = 0,
        n_labels: int = 0,
        n_hidden: int = 128,
        n_latent: int = 10,
        n_code: int = 50,
        G = None, # GENERATOR MATRIX G
        n_layers: int = 1,
        n_continuous_cov: int = 0,
        n_cats_per_cov: list[int] | None = None,
        dropout_rate: float = 0.1,
        beta: int = 10, # ADD TEMPERATURE VARIABLE
        dispersion: Literal["gene", "gene-batch", "gene-label", "gene-cell"] = "gene",
        log_variational: bool = True,
        gene_likelihood: Literal["zinb", "nb", "poisson"] = "zinb",
        latent_distribution: Literal["normal", "ln", "ber"] = "ber", # ADD DVAE OPTION
        coded_key: bool = False, # ADD CODED / UNCODED FLAG
        encode_covariates: bool = False,
        deeply_inject_covariates: bool = True,
        batch_representation: Literal["one-hot", "embedding"] = "one-hot",
        use_batch_norm: Literal["encoder", "decoder", "none", "both"] = "both",
        use_layer_norm: Literal["encoder", "decoder", "none", "both"] = "none",
        use_size_factor_key: bool = False,
        use_observed_lib_size: bool = True,
        extra_payload_autotune: bool = False,
        library_log_means: np.ndarray | None = None,
        library_log_vars: np.ndarray | None = None,
        var_activation: Callable[[torch.Tensor], torch.Tensor] = None,
        extra_encoder_kwargs: dict | None = None,
        extra_decoder_kwargs: dict | None = None,
        batch_embedding_kwargs: dict | None = None,
    ):
        from scvi.nn import DecoderSCVI
        # https://docs.scvi-tools.org/en/stable/api/reference/scvi.nn.Encoder.html

        super().__init__()

        self.dispersion = dispersion
        self.n_latent = n_latent
        self.beta = beta
        self.log_variational = log_variational
        self.gene_likelihood = gene_likelihood
        self.n_batch = n_batch
        self.n_labels = n_labels

        # MODIFIED ATTRIBUTES
        self.latent_distribution = latent_distribution
        self.coded_key = coded_key
        self.G = G # Not used here, only by encoder

        # METADATA
        self.encode_covariates = encode_covariates
        self.use_size_factor_key = use_size_factor_key
        self.use_observed_lib_size = use_size_factor_key or use_observed_lib_size
        self.extra_payload_autotune = extra_payload_autotune

        if not self.use_observed_lib_size:
            if library_log_means is None or library_log_vars is None:
                raise ValueError(
                    "If not using observed_lib_size, "
                    "must provide library_log_means and library_log_vars."
                )

            self.register_buffer("library_log_means", torch.from_numpy(library_log_means).float())
            self.register_buffer("library_log_vars", torch.from_numpy(library_log_vars).float())

        if self.dispersion == "gene":
            self.px_r = torch.nn.Parameter(torch.randn(n_input))
        elif self.dispersion == "gene-batch":
            self.px_r = torch.nn.Parameter(torch.randn(n_input, n_batch))
        elif self.dispersion == "gene-label":
            self.px_r = torch.nn.Parameter(torch.randn(n_input, n_labels))
        elif self.dispersion == "gene-cell":
            pass
        else:
            raise ValueError(
                "`dispersion` must be one of 'gene', 'gene-batch', 'gene-label', 'gene-cell'."
            )

        self.batch_representation = batch_representation
        if self.batch_representation == "embedding":
            self.init_embedding(REGISTRY_KEYS.BATCH_KEY, n_batch, **(batch_embedding_kwargs or {}))
            batch_dim = self.get_embedding(REGISTRY_KEYS.BATCH_KEY).embedding_dim
        elif self.batch_representation != "one-hot":
            raise ValueError("`batch_representation` must be one of 'one-hot', 'embedding'.")

        use_batch_norm_encoder = use_batch_norm == "encoder" or use_batch_norm == "both"
        use_batch_norm_decoder = use_batch_norm == "decoder" or use_batch_norm == "both"
        use_layer_norm_encoder = use_layer_norm == "encoder" or use_layer_norm == "both"
        use_layer_norm_decoder = use_layer_norm == "decoder" or use_layer_norm == "both"

        n_input_encoder = n_input + n_continuous_cov * encode_covariates
        z_encoder_out = n_code if coded_key else n_latent
        if self.batch_representation == "embedding":
            n_input_encoder += batch_dim * encode_covariates
            cat_list = list([] if n_cats_per_cov is None else n_cats_per_cov)
        else:
            cat_list = [n_batch] + list([] if n_cats_per_cov is None else n_cats_per_cov)

        encoder_cat_list = cat_list if encode_covariates else None
        _extra_encoder_kwargs = extra_encoder_kwargs or {}
        self.z_encoder = Encoder(
            n_input_encoder,
            z_encoder_out,
            n_cat_list=encoder_cat_list,
            n_layers=n_layers,
            n_hidden=n_hidden,
            dropout_rate=dropout_rate,
            distribution=latent_distribution,
            coded_key=coded_key,
            G=G,
            inject_covariates=deeply_inject_covariates,
            use_batch_norm=use_batch_norm_encoder,
            use_layer_norm=use_layer_norm_encoder,
            var_activation=var_activation,
            return_dist=True,
            **_extra_encoder_kwargs,
        )
        # l encoder goes from n_input-dimensional data to 1-d library size
        self.l_encoder = Encoder(
            n_input_encoder,
            1,
            n_layers=1,
            n_cat_list=encoder_cat_list,
            n_hidden=n_hidden,
            dropout_rate=dropout_rate,
            distribution="normal",
            inject_covariates=deeply_inject_covariates,
            use_batch_norm=use_batch_norm_encoder,
            use_layer_norm=use_layer_norm_encoder,
            var_activation=var_activation,
            return_dist=True,
            **_extra_encoder_kwargs,
        )

        n_input_decoder = (n_code if coded_key else n_latent) + n_continuous_cov
        if self.batch_representation == "embedding":
            n_input_decoder += batch_dim

        _extra_decoder_kwargs = extra_decoder_kwargs or {}
        self.decoder = DecoderSCVI(
            n_input_decoder,
            n_input,
            n_cat_list=cat_list,
            n_layers=n_layers,
            n_hidden=n_hidden,
            inject_covariates=deeply_inject_covariates,
            use_batch_norm=use_batch_norm_decoder,
            use_layer_norm=use_layer_norm_decoder,
            scale_activation="softplus" if use_size_factor_key else "softmax",
            **_extra_decoder_kwargs,
        )

    def _get_inference_input(
        self,
        tensors: dict[str, torch.Tensor | None],
        full_forward_pass: bool = False,
    ) -> dict[str, torch.Tensor | None]:
        """Get input tensors for the inference process."""
        if full_forward_pass or self.minified_data_type is None:
            loader = "full_data"
        elif self.minified_data_type in [
            ADATA_MINIFY_TYPE.LATENT_POSTERIOR,
            ADATA_MINIFY_TYPE.LATENT_POSTERIOR_WITH_COUNTS,
        ]:
            loader = "minified_data"
        else:
            raise NotImplementedError(f"Unknown minified-data type: {self.minified_data_type}")

        if loader == "full_data":
            return {
                MODULE_KEYS.X_KEY: tensors[REGISTRY_KEYS.X_KEY],
                MODULE_KEYS.BATCH_INDEX_KEY: tensors[REGISTRY_KEYS.BATCH_KEY],
                MODULE_KEYS.CONT_COVS_KEY: tensors.get(REGISTRY_KEYS.CONT_COVS_KEY, None),
                MODULE_KEYS.CAT_COVS_KEY: tensors.get(REGISTRY_KEYS.CAT_COVS_KEY, None),
            }
        else:
            # if self.latent_distribution == "ber":
            #     return{
            #         MODULE_KEYS.QZ_KEY: tensors[REGISTRY_KEYS.LATENT_QZ_KEY],
            #         REGISTRY_KEYS.OBSERVED_LIB_SIZE: tensors[REGISTRY_KEYS.OBSERVED_LIB_SIZE],
            #     }
            # else:
            return {
                MODULE_KEYS.QZM_KEY: tensors[REGISTRY_KEYS.LATENT_QZM_KEY],
                MODULE_KEYS.QZV_KEY: tensors[REGISTRY_KEYS.LATENT_QZV_KEY],
                REGISTRY_KEYS.OBSERVED_LIB_SIZE: tensors[REGISTRY_KEYS.OBSERVED_LIB_SIZE],
            }

    def _get_generative_input(
        self,
        tensors: dict[str, torch.Tensor],
        inference_outputs: dict[str, torch.Tensor | Distribution | None],
    ) -> dict[str, torch.Tensor | None]:
        """Get input tensors for the generative process."""
        size_factor = tensors.get(REGISTRY_KEYS.SIZE_FACTOR_KEY, None)
        if size_factor is not None:
            size_factor = torch.log(size_factor)

        return {
            MODULE_KEYS.Z_KEY: inference_outputs[MODULE_KEYS.Z_KEY],
            MODULE_KEYS.LIBRARY_KEY: inference_outputs[MODULE_KEYS.LIBRARY_KEY],
            MODULE_KEYS.BATCH_INDEX_KEY: tensors[REGISTRY_KEYS.BATCH_KEY],
            MODULE_KEYS.Y_KEY: tensors[REGISTRY_KEYS.LABELS_KEY],
            MODULE_KEYS.CONT_COVS_KEY: tensors.get(REGISTRY_KEYS.CONT_COVS_KEY, None),
            MODULE_KEYS.CAT_COVS_KEY: tensors.get(REGISTRY_KEYS.CAT_COVS_KEY, None),
            MODULE_KEYS.SIZE_FACTOR_KEY: size_factor,
        }

    def _compute_local_library_params(
        self,
        batch_index: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Computes local library parameters.

        Compute two tensors of shape (batch_index.shape[0], 1) where each
        element corresponds to the mean and variances, respectively, of the
        log library sizes in the batch the cell corresponds to.
        """
        from torch.nn.functional import linear

        n_batch = self.library_log_means.shape[1]
        local_library_log_means = linear(
            one_hot(batch_index.squeeze(-1), n_batch).float(), self.library_log_means
        )

        local_library_log_vars = linear(
            one_hot(batch_index.squeeze(-1), n_batch).float(), self.library_log_vars
        )

        return local_library_log_means, local_library_log_vars

    @auto_move_data
    def _regular_inference(
        self,
        x: torch.Tensor,
        batch_index: torch.Tensor,
        cont_covs: torch.Tensor | None = None,
        cat_covs: torch.Tensor | None = None,
        n_samples: int = 1,
    ) -> dict[str, torch.Tensor | Distribution | None]:
        """Run the regular inference process."""
        x_ = x
        if self.use_observed_lib_size:
            library = torch.log(x.sum(1)).unsqueeze(1)
        if self.log_variational:
            x_ = torch.log1p(x_)

        if cont_covs is not None and self.encode_covariates:
            encoder_input = torch.cat((x_, cont_covs), dim=-1)
        else:
            encoder_input = x_
        if cat_covs is not None and self.encode_covariates:
            categorical_input = torch.split(cat_covs, 1, dim=1)
        else:
            categorical_input = ()

        if self.latent_distribution == "ber":
            # Encoder returns 3 values: (q_z, q_m, z)
            # q_z: n_latent (uncoded) or n_code (coded) - bit probs for decoder
            # q_m: n_latent - bit probabilities for KL
            # z: n_latent (uncoded) or n_code (coded) - samples for decoder
            qz, qm, z = self.z_encoder(encoder_input, batch_index, *categorical_input)
        else:
            if self.batch_representation == "embedding" and self.encode_covariates:
                batch_rep = self.compute_embedding(REGISTRY_KEYS.BATCH_KEY, batch_index)
                encoder_input = torch.cat([encoder_input, batch_rep], dim=-1)
                qz, z = self.z_encoder(encoder_input, *categorical_input)
            else:
                qz, z = self.z_encoder(encoder_input, batch_index, *categorical_input)
            qm = None # This variable is not used for non-bernoulli latent dist

        ql = None
        if not self.use_observed_lib_size:
            if self.batch_representation == "embedding":
                ql, library_encoded = self.l_encoder(encoder_input, *categorical_input)
            else:
                ql, library_encoded = self.l_encoder(
                    encoder_input, batch_index, *categorical_input
                )
            library = library_encoded

        if n_samples > 1:
            if self.latent_distribution == "ber":
                # TODO: Multiple samples not supported.
                # z is single sample, and this happens within the encoder
                # Repeating z is not independent sampling process ---
                raise NotImplementedError("Not implemented for discrete latent space.")
            else:
                untran_z = qz.sample((n_samples,))
                z = self.z_encoder.z_transformation(untran_z)
                if self.use_observed_lib_size:
                    library = library.unsqueeze(0).expand(
                        (n_samples, library.size(0), library.size(1))
                    )
                else:
                    library = ql.sample((n_samples,))

        return {
            MODULE_KEYS.Z_KEY: z,
            MODULE_KEYS.QZ_KEY: qz,
            # IN UNCODED CASE, QZ IS P(Z|M)
            # IN CODED CASE, QZ IS P(Z|C)
            "Q_M": qm,  # n_latent bit probabilities, None if continuous
            MODULE_KEYS.QL_KEY: ql,
            MODULE_KEYS.LIBRARY_KEY: library,
        }

    # def _cached_inference(
    #     self,
    #     qzm: torch.Tensor,
    #     qzv: torch.Tensor,
    #     observed_lib_size: torch.Tensor,
    #     n_samples: int = 1,
    # ) -> dict[str, torch.Tensor | None]:
    #     """Run the cached inference process."""
    #     from torch.distributions import Normal

    #     qz = Normal(qzm, qzv.sqrt())
    #     # use dist.sample() rather than rsample because we aren't optimizing the z here
    #     untran_z = qz.sample() if n_samples == 1 else qz.sample((n_samples,))
    #     z = self.z_encoder.z_transformation(untran_z)
    #     library = torch.log(observed_lib_size)
    #     if n_samples > 1:
    #         library = library.unsqueeze(0).expand((n_samples, library.size(0), library.size(1)))

    #     return {
    #         MODULE_KEYS.Z_KEY: z,
    #         MODULE_KEYS.QZ_KEY: qz,
    #         MODULE_KEYS.QL_KEY: None,
    #         MODULE_KEYS.LIBRARY_KEY: library,
    #     }

    @auto_move_data
    def generative(
        self,
        z: torch.Tensor,
        library: torch.Tensor,
        batch_index: torch.Tensor,
        cont_covs: torch.Tensor | None = None,
        cat_covs: torch.Tensor | None = None,
        size_factor: torch.Tensor | None = None,
        y: torch.Tensor | None = None,
        transform_batch: torch.Tensor | None = None,
    ) -> dict[str, Distribution | None]:
        """Run the generative process."""
        from torch.nn.functional import linear

        from scvi.distributions import (
            NegativeBinomial,
            Normal,
            Poisson,
            ZeroInflatedNegativeBinomial,
        )

        # TODO: refactor forward function to not rely on y
        # Likelihood distribution
        if cont_covs is None:
            decoder_input = z
        elif z.dim() != cont_covs.dim():
            decoder_input = torch.cat(
                [z, cont_covs.unsqueeze(0).expand(z.size(0), -1, -1)], dim=-1
            )
        else:
            decoder_input = torch.cat([z, cont_covs], dim=-1)

        if cat_covs is not None:
            categorical_input = torch.split(cat_covs, 1, dim=1)
        else:
            categorical_input = ()

        if transform_batch is not None:
            batch_index = torch.ones_like(batch_index) * transform_batch

        if not self.use_size_factor_key:
            size_factor = library

        if self.batch_representation == "embedding":
            batch_rep = self.compute_embedding(REGISTRY_KEYS.BATCH_KEY, batch_index)
            decoder_input = torch.cat([decoder_input, batch_rep], dim=-1)
            px_scale, px_r, px_rate, px_dropout = self.decoder(
                self.dispersion,
                decoder_input,
                size_factor,
                *categorical_input,
                y,
            )
        else:
            px_scale, px_r, px_rate, px_dropout = self.decoder(
                self.dispersion,
                decoder_input,
                size_factor,
                batch_index,
                *categorical_input,
                y,
            )

        if self.dispersion == "gene-label":
            px_r = linear(
                one_hot(y.squeeze(-1), self.n_labels).float(), self.px_r
            )  # px_r gets transposed - last dimension is nb genes
        elif self.dispersion == "gene-batch":
            px_r = linear(one_hot(batch_index.squeeze(-1), self.n_batch).float(), self.px_r)
        elif self.dispersion == "gene":
            px_r = self.px_r

        px_r = torch.exp(px_r)

        if self.gene_likelihood == "zinb":
            px = ZeroInflatedNegativeBinomial(
                mu=px_rate,
                theta=px_r,
                zi_logits=px_dropout,
                scale=px_scale,
            )
        elif self.gene_likelihood == "nb":
            px = NegativeBinomial(mu=px_rate, theta=px_r, scale=px_scale)
        elif self.gene_likelihood == "poisson":
            px = Poisson(rate=px_rate, scale=px_scale)
        elif self.gene_likelihood == "normal":
            px = Normal(px_rate, px_r, normal_mu=px_scale)

        # Priors
        if self.use_observed_lib_size:
            pl = None
        else:
            (
                local_library_log_means,
                local_library_log_vars,
            ) = self._compute_local_library_params(batch_index)
            pl = Normal(local_library_log_means, local_library_log_vars.sqrt())

        # ADDITION: HANDLE PZ FOR DISCRETE
        # https://github.com/mariamartinezgarcia/codedVAE/blob/main/src/coded_vae.py#L345
        # pz = torch.ones((1, self.n_latent))*0.5
        if self.latent_distribution == "ber":
            # Prior: independent Bernoulli(0.5) for each bit
            # Return a tensor of prior probabilities (broadcastable to batch size)
            pz = torch.full((1, self.n_latent), 0.5, device=z.device)
        else:
            pz = Normal(torch.zeros_like(z), torch.ones_like(z))

        return {
            MODULE_KEYS.PX_KEY: px,
            MODULE_KEYS.PL_KEY: pl,
            MODULE_KEYS.PZ_KEY: pz,
        }

    @unsupported_if_adata_minified
    def loss(
        self,
        tensors: dict[str, torch.Tensor],
        inference_outputs: dict[str, torch.Tensor | Distribution | None],
        generative_outputs: dict[str, Distribution | None],
        kl_weight: torch.tensor | float = 1.0,
    ) -> LossOutput:
        """Compute the loss."""
        from torch.distributions import kl_divergence
        from torch.distributions import Bernoulli # prior

        x = tensors[REGISTRY_KEYS.X_KEY]
        pz = generative_outputs[MODULE_KEYS.PZ_KEY]

        # ADDITION: KL DIVERGENCE FOR DISCRETE
        # https://github.com/mariamartinezgarcia/codedVAE/blob/main/src/train/loss.py#L139
        if self.latent_distribution == "ber":
            qm = inference_outputs["Q_M"]   # qm has n_latent-dim bit probabilities for both uncoded/coded
            kl_divergence_z = kl_div_bernoulli(qm, pz)
        else:
            qz = inference_outputs[MODULE_KEYS.QZ_KEY]
            kl_divergence_z = kl_divergence(qz, pz).sum(dim=-1)


        if not self.use_observed_lib_size:
            kl_divergence_l = kl_divergence(
                inference_outputs[MODULE_KEYS.QL_KEY], generative_outputs[MODULE_KEYS.PL_KEY]
            ).sum(dim=1)
        else:
            kl_divergence_l = torch.zeros_like(kl_divergence_z)

        reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)

        kl_local_for_warmup = kl_divergence_z
        kl_local_no_warmup = kl_divergence_l

        weighted_kl_local = kl_weight * kl_local_for_warmup + kl_local_no_warmup

        loss = torch.mean(reconst_loss + weighted_kl_local)

        # a payload to be used during autotune
        if self.extra_payload_autotune:
            extra_metrics_payload = {
                "z": inference_outputs["z"],
                "batch": tensors[REGISTRY_KEYS.BATCH_KEY],
                "labels": tensors[REGISTRY_KEYS.LABELS_KEY],
            }
        else:
            extra_metrics_payload = {}

        return LossOutput(
            loss=loss,
            reconstruction_loss=reconst_loss,
            kl_local={
                MODULE_KEYS.KL_L_KEY: kl_divergence_l,
                MODULE_KEYS.KL_Z_KEY: kl_divergence_z,
            },
            extra_metrics=extra_metrics_payload,
        )

    @torch.inference_mode()
    def sample(
        self,
        tensors: dict[str, torch.Tensor],
        n_samples: int = 1,
        max_poisson_rate: float = 1e8,
        generative_kwargs: dict | None = None,
    ) -> torch.Tensor:
        r"""Generate predictive samples from the posterior predictive distribution.

        The posterior predictive distribution is denoted as :math:`p(\hat{x} \mid x)`, where
        :math:`x` is the input data and :math:`\hat{x}` is the sampled data.

        We sample from this distribution by first sampling ``n_samples`` times from the posterior
        distribution :math:`q(z \mid x)` for a given observation, and then sampling from the
        likelihood :math:`p(\hat{x} \mid z)` for each of these.

        Parameters
        ----------
        tensors
            Dictionary of tensors passed into :meth:`~scvi.module.VAE.forward`.
        n_samples
            Number of Monte Carlo samples to draw from the distribution for each observation.
        max_poisson_rate
            The maximum value to which to clip the ``rate`` parameter of
            :class:`~scvi.distributions.Poisson`. Avoids numerical sampling issues when the
            parameter is very large due to the variance of the distribution.
        generative_kwargs
            Keyword args for ``generative()`` in fwd pass

        Returns
        -------
        Tensor on CPU with shape ``(n_obs, n_vars)`` if ``n_samples == 1``, else
        ``(n_obs, n_vars, n_samples)``.
        """
        from scvi.distributions import Poisson
        from torch.distributions import Bernoulli

        if self.latent_distribution == "ber":
            # Get posterior bit probabilities from single forward pass
            inference_outputs, _ = self.forward(
                tensors,
                inference_kwargs={"n_samples": 1},
                generative_kwargs=generative_kwargs,
                compute_loss=False,
            )

            qz = inference_outputs[MODULE_KEYS.QZ_KEY]  # (batch_size, n_latent/n_code)
            batch_size = qz.shape[0]
            # CUSTOM REPARAMETRIZATION FOR SAMPLING
            beta=torch.tensor(self.beta, device=qz.device)
            z_samples = sample_from_qz_given_x(qz, beta=beta, n_samples=n_samples) # (batch, n_latent/n_code, n_samples)

            library = inference_outputs[MODULE_KEYS.LIBRARY_KEY]  # (batch_size, 1)
            batch_index = tensors[REGISTRY_KEYS.BATCH_KEY]  # (batch_size, 1)
            cont_covs = tensors.get(REGISTRY_KEYS.CONT_COVS_KEY, None)
            cat_covs = tensors.get(REGISTRY_KEYS.CAT_COVS_KEY, None)
            size_factor = tensors.get(REGISTRY_KEYS.SIZE_FACTOR_KEY, None)
            y = tensors.get(REGISTRY_KEYS.LABELS_KEY, None)

            if n_samples > 1:
                z = z_samples.permute(0,2,1).reshape(-1,z_samples.shape[1])
                batch_index = batch_index.unsqueeze(1).expand(-1, n_samples, -1).reshape(-1, 1)  # (n_samples, batch_size, 1)
                library = library.unsqueeze(1).expand(-1, n_samples, -1).reshape(-1, 1)   # (n_samples, batch_size, 1)
                if cont_covs is not None:
                    cont_covs = cont_covs.unsqueeze(1).expand(-1, n_samples, -1).reshape(-1, cont_covs.shape[-1])
                if cat_covs is not None:
                    cat_covs = cat_covs.unsqueeze(1).expand(-1, n_samples, -1).reshape(-1, cat_covs.shape[-1])
                if size_factor is not None:
                    size_factor = size_factor.unsqueeze(1).expand(-1, n_samples, -1).reshape(-1, 1)
                if y is not None:
                    y = y.unsqueeze(1).expand(-1, n_samples, -1).reshape(-1, 1)
            else:
                z = z_samples.squeeze(-1) # (batch_size, n_latent/n_code)


            # Single generative call - let it handle multi-dimensional shapes
            generative_outputs = self.generative(
                z=z,
                library=library,
                batch_index=batch_index,
                cont_covs=cont_covs,
                cat_covs=cat_covs,
                size_factor=size_factor,
                y=y,
                transform_batch=None,
            )

            dist = generative_outputs[MODULE_KEYS.PX_KEY]
            if self.gene_likelihood == "poisson":
                dist = (
                    Poisson(torch.clamp(dist.rate.to("cpu"), max=max_poisson_rate))
                    if self.device.type == "mps"
                    else Poisson(torch.clamp(dist.rate, max=max_poisson_rate))
                )

            samples = dist.sample()
            if n_samples > 1:
                # samples shape: (n_samples, batch_size, n_vars)
                # permute to: (batch_size, n_vars, n_samples)
                samples = samples.reshape(batch_size, n_samples, -1).permute(0, 2, 1)

        else:
            # GAUSSIAN LATENT SPACE - UNCHANGED FROM ORIGINAL
            inference_kwargs = {"n_samples": n_samples}
            _, generative_outputs = self.forward(
                tensors,
                inference_kwargs=inference_kwargs,
                generative_kwargs=generative_kwargs,
                compute_loss=False,
            )

            dist = generative_outputs[MODULE_KEYS.PX_KEY]
            if self.gene_likelihood == "poisson":
                dist = (
                    Poisson(torch.clamp(dist.rate.to("cpu"), max=max_poisson_rate))
                    if self.device.type == "mps"
                    else Poisson(torch.clamp(dist.rate, max=max_poisson_rate))
                )

            samples = dist.sample()
            samples = torch.permute(samples, (1, 2, 0)) if n_samples > 1 else samples

        return samples.cpu()



    # NOT IMPLEMENTED YET
    @torch.inference_mode()
    @auto_move_data
    def marginal_ll(
        self,
        tensors: dict[str, torch.Tensor],
        n_mc_samples: int,
        return_mean: bool = False,
        n_mc_samples_per_pass: int = 1,
    ):
        """Compute the marginal log-likelihood of the data under the model.

        Parameters
        ----------
        tensors
            Dictionary of tensors passed into :meth:`~scvi.module.VAE.forward`.
        n_mc_samples
            Number of Monte Carlo samples to use for the estimation of the marginal log-likelihood.
        return_mean
            Whether to return the mean of marginal likelihoods over cells.
        n_mc_samples_per_pass
            Number of Monte Carlo samples to use per pass. This is useful to avoid memory issues.
        """
        if self.latent_distribution == "ber":
            raise NotImplementedError("Not implemented for Bernoulli latent distribution.")
            # marginal_ll requires log-prob evaluation under prior and approximate posterior
            # use loss metrics to compare models instead

        from torch import logsumexp
        from torch.distributions import Normal

        batch_index = tensors[REGISTRY_KEYS.BATCH_KEY]

        to_sum = []
        if n_mc_samples_per_pass > n_mc_samples:
            warnings.warn(
                "Number of chunks is larger than the total number of samples, setting it to the "
                "number of samples",
                RuntimeWarning,
                stacklevel=settings.warnings_stacklevel,
            )
            n_mc_samples_per_pass = n_mc_samples
        n_passes = int(np.ceil(n_mc_samples / n_mc_samples_per_pass))
        for _ in range(n_passes):
            # Distribution parameters and sampled variables
            inference_outputs, _, losses = self.forward(
                tensors,
                inference_kwargs={"n_samples": n_mc_samples_per_pass},
                get_inference_input_kwargs={"full_forward_pass": True},
            )
            qz = inference_outputs[MODULE_KEYS.QZ_KEY]
            ql = inference_outputs[MODULE_KEYS.QL_KEY]
            z = inference_outputs[MODULE_KEYS.Z_KEY]
            library = inference_outputs[MODULE_KEYS.LIBRARY_KEY]

            # Reconstruction Loss
            reconst_loss = losses.dict_sum(losses.reconstruction_loss)

            # Log-probabilities
            p_z = (
                Normal(torch.zeros_like(qz.loc), torch.ones_like(qz.scale)).log_prob(z).sum(dim=-1)
            )
            p_x_zl = -reconst_loss
            q_z_x = qz.log_prob(z).sum(dim=-1)
            log_prob_sum = p_z + p_x_zl - q_z_x

            if not self.use_observed_lib_size:
                (
                    local_library_log_means,
                    local_library_log_vars,
                ) = self._compute_local_library_params(batch_index)

                p_l = (
                    Normal(local_library_log_means, local_library_log_vars.sqrt())
                    .log_prob(library)
                    .sum(dim=-1)
                )
                q_l_x = ql.log_prob(library).sum(dim=-1)

                log_prob_sum += p_l - q_l_x
            if n_mc_samples_per_pass == 1:
                log_prob_sum = log_prob_sum.unsqueeze(0)

            to_sum.append(log_prob_sum)
        to_sum = torch.cat(to_sum, dim=0)
        batch_log_lkl = logsumexp(to_sum, dim=0) - np.log(n_mc_samples)
        if return_mean:
            batch_log_lkl = torch.mean(batch_log_lkl).item()
        else:
            batch_log_lkl = batch_log_lkl.cpu()
        return batch_log_lkl


class DSCVI(RNASeqMixin, VAEMixin, ArchesMixin, UnsupervisedTrainingMixin, BaseMinifiedModeModelClass):
    """
    scVI-style model for discrete uncoded VAE

    Parameters
    ----------
    adata: AnnData object that has been registered via `~scvi.model.SCVI.setup_anndata`.
    n_latent: Dimensionality of the latent space. For uncoded scvi-vae it is m (information bits)

    n_hidden: Number of nodes per hidden layer.
    n_layers: Number of hidden layers used for encoder and decoder NNs.
    dropout_rate: Dropout rate for neural networks.
    beta:

    dispersion
        * 'gene' - dispersion parameter of NB is constant per gene across cells
        * 'gene-batch' - dispersion can differ between different batches
        * 'gene-label' - dispersion can differ between different labels
        * 'gene-cell' - dispersion can differ for every gene in every cell
    gene_likelihood
        * 'nb' - Negative binomial distribution
        * 'zinb' - Zero-inflated negative binomial distribution
        * 'poisson' - Poisson distribution
    latent_distribution
        * 'normal' - Normal distribution
        * 'ln' - Logistic normal distribution (Normal(0, I) transformed by softmax)
    **kwargs: Additional keyword arguments for :class:`~scvi.module.VAE`.
    """

    _module_cls = DSCVIModule # Custom implementation for a module
    _training_plan_cls = TrainingPlan  # default; gridsearch cells override this per-instance


    def __init__(
        self,
        adata,
        n_latent: int = 10,
        n_code: int = 50,
        G: torch.Tensor = None,
        n_hidden: int = 128,
        n_layers: int = 1,
        dropout_rate: float = 0.1,
        beta: int = 10,
        dispersion: Literal["gene", "gene-batch", "gene-label", "gene-cell"] = "gene",
        gene_likelihood: Literal["zinb", "nb", "poisson"] = "zinb",
        latent_distribution: Literal["normal", "ln", "ber"] = "ber", # Added option for discrete - ber
        coded_key: bool = False, # We add a flag to distinguish coded / uncoded behaviour
        **kwargs,
    ):
        super().__init__(adata)


        self._module_kwargs = {
            "n_hidden": n_hidden,
            "n_latent": n_latent,
            "n_layers": n_layers,
            "dropout_rate": dropout_rate,
            "beta": beta,
            "dispersion": dispersion,
            "gene_likelihood": gene_likelihood,
            "latent_distribution": latent_distribution,
            "coded_key": coded_key,
            **kwargs,
        }
        self._model_summary_string = (
            "DSCVI model with the following parameters: \n"
            f"n_hidden: {n_hidden}, n_latent: {n_latent}, n_layers: {n_layers}, "
            f"dropout_rate: {dropout_rate}, beta: {beta}, dispersion: {dispersion}, "
            f"gene_likelihood: {gene_likelihood}, latent_distribution: {latent_distribution}, "
            f"coded_key: {coded_key}"
        )

        if self.minified_data_type is not None:
            raise NotImplementedError("Minified mode not supported for discrete latent space")


        if self._module_init_on_train:
            self.module = None
            warnings.warn(
                "Model was initialized without `adata`. The module will be initialized when "
                "calling `train`. This behavior is experimental and may change in the future.",
                UserWarning,
                stacklevel=settings.warnings_stacklevel,
            )
        else:
            n_cats_per_cov = (
                self.adata_manager.get_state_registry(REGISTRY_KEYS.CAT_COVS_KEY).n_cats_per_key
                if REGISTRY_KEYS.CAT_COVS_KEY in self.adata_manager.data_registry
                else None
            )
            n_batch = self.summary_stats.n_batch
            use_size_factor_key = REGISTRY_KEYS.SIZE_FACTOR_KEY in self.adata_manager.data_registry
            library_log_means, library_log_vars = None, None
            if not use_size_factor_key and self.minified_data_type is None:
                library_log_means, library_log_vars = _init_library_size(
                    self.adata_manager, n_batch
                )
            self.module = self._module_cls(
                n_input=self.summary_stats.n_vars,
                n_batch=n_batch,
                n_labels=self.summary_stats.n_labels,
                n_continuous_cov=self.summary_stats.get("n_extra_continuous_covs", 0),
                n_cats_per_cov=n_cats_per_cov,
                n_hidden=n_hidden,
                n_latent=n_latent,
                n_code=n_code,
                G=G,
                coded_key=coded_key,
                n_layers=n_layers,
                dropout_rate=dropout_rate,
                beta=beta,
                dispersion=dispersion,
                gene_likelihood=gene_likelihood,
                latent_distribution=latent_distribution,
                use_size_factor_key=use_size_factor_key,
                library_log_means=library_log_means,
                library_log_vars=library_log_vars,
                **kwargs,
            )


        self.init_params_ = self._get_init_params(locals())


    @classmethod
    def setup_anndata(
        cls,
        adata: AnnData,
        layer: str | None = None,
        batch_key: str | None = None,
        labels_key: str | None = None,
        size_factor_key: str | None = None,
        categorical_covariate_keys: list[str] | None = None,
        continuous_covariate_keys: list[str] | None = None,
        **kwargs,
    ):
        """%(summary)s.

        Parameters
        ----------
        %(param_adata)s
        %(param_layer)s
        %(param_batch_key)s
        %(param_labels_key)s
        %(param_size_factor_key)s
        %(param_cat_cov_keys)s
        %(param_cont_cov_keys)s
        """
        setup_method_args = cls._get_setup_method_args(**locals())
        anndata_fields = [
            LayerField(REGISTRY_KEYS.X_KEY, layer, is_count_data=True),
            CategoricalObsField(REGISTRY_KEYS.BATCH_KEY, batch_key),
            CategoricalObsField(REGISTRY_KEYS.LABELS_KEY, labels_key),
            NumericalObsField(REGISTRY_KEYS.SIZE_FACTOR_KEY, size_factor_key, required=False),
            CategoricalJointObsField(REGISTRY_KEYS.CAT_COVS_KEY, categorical_covariate_keys),
            NumericalJointObsField(REGISTRY_KEYS.CONT_COVS_KEY, continuous_covariate_keys),
        ]
        adata_manager = AnnDataManager(fields=anndata_fields, setup_method_args=setup_method_args)
        adata_manager.register_fields(adata, **kwargs)
        cls.register_manager(adata_manager)


    @torch.inference_mode()
    def get_latent_representation(
        self,
        adata=None,
        indices=None,
        give_mean: bool = True,
        mc_samples: int = 5000,
        batch_size=None,
        return_dist: bool = False,
    ):
        self._check_if_trained(warn=False)
        adata = self._validate_anndata(adata)
        scdl = self._make_data_loader(adata=adata, indices=indices, batch_size=batch_size)

        latent = []
        latent_qzm = []
        latent_qzv = []

        is_discrete = (hasattr(self.module, 'latent_distribution') and
                      self.module.latent_distribution == "ber")

        for tensors in scdl:
            inference_inputs = self.module._get_inference_input(tensors)
            outputs = self.module.inference(**inference_inputs)

            if is_discrete:
                # q_m: M-dim message-bit probabilities (both coded and uncoded)
                qm = outputs["Q_M"]

                if return_dist:
                    # Return (mean, variance) of the Bernoulli over message bits
                    mean = qm
                    var = qm * (1 - qm)
                    latent_qzm.append(mean.cpu())
                    latent_qzv.append(var.cpu())

                elif give_mean:
                    # Mean of Bernoulli = probability itself
                    latent.append(qm.cpu())

                else:
                    # Return the sampled latent z (D-dim if coded, M-dim if uncoded)
                    latent.append(outputs[MODULE_KEYS.Z_KEY].cpu())

            else:
                # Continuous case (unchanged)
                qz = outputs.get(MODULE_KEYS.QZ_KEY, None)
                if qz is None:
                    qz_m, qz_v = outputs["qz_m"], outputs["qz_v"]
                    qz = torch.distributions.Normal(qz_m, qz_v.sqrt())

                if return_dist:
                    latent_qzm.append(qz.loc.cpu())
                    latent_qzv.append(qz.scale.square().cpu())

                elif give_mean:
                    if self.module.latent_distribution == "ln":
                        samples = qz.sample([mc_samples])
                        z = torch.nn.functional.softmax(samples, dim=-1)
                        z = z.mean(dim=0)
                    else:
                        z = qz.loc
                    latent.append(z.cpu())

                else:
                    latent.append(outputs[MODULE_KEYS.Z_KEY].cpu())

        # Return AFTER loop — all batches accumulated
        if return_dist:
            return (torch.cat(latent_qzm).numpy(), torch.cat(latent_qzv).numpy())
        else:
            return torch.cat(latent).numpy()


    def train(
        self,
        max_epochs: int | None = None,
        accelerator: str = "auto",
        devices: int | list[int] | str = "auto",
        train_size: float = 0.9,
        validation_size: float | None = None,
        shuffle_set_split: bool = True,
        batch_size: int = 128,
        early_stopping: bool = False,
        plan_kwargs: dict | None = None,
        **trainer_kwargs,
    ):
        """
        Train the model.

        Parameters
        ----------
        max_epochs : int, optional
            Maximum number of epochs. If None, a heuristic based on dataset size is used.
        accelerator : str
            "cpu", "gpu", "tpu", etc.
        devices : int, list, or str
            Number of devices or device indices.
        train_size : float
            Fraction of data used for training (0.0 - 1.0).
        validation_size : float, optional
            Fraction of data used for validation. If None, uses 1 - train_size.
        shuffle_set_split : bool
            Whether to shuffle data before splitting.
        batch_size : int
            Minibatch size.
        early_stopping : bool
            If True, stops training when validation loss stops improving.
        plan_kwargs : dict, optional
            Extra arguments passed to the TrainingPlan (e.g., learning rate).
        **trainer_kwargs
            Additional arguments for the PyTorch Lightning Trainer.
        """
        # ----- 1. Determine max_epochs if not provided -----
        if max_epochs is None:
            max_epochs = get_max_epochs_heuristic(self.adata.n_obs)

        # ----- 2. Create the data splitter -----
        data_module = DataSplitter(
            self.adata_manager,
            train_size=train_size,
            validation_size=validation_size,
            batch_size=batch_size,
            shuffle_set_split=shuffle_set_split,
        )

        # ----- 3. Create the training plan -----
        plan_kwargs = plan_kwargs or {}
        # training_plan = TrainingPlan(self.module, **plan_kwargs)
        training_plan = self._training_plan_cls(self.module, **plan_kwargs)

        # ----- 4. Ensure early_stopping is passed correctly -----
        if "early_stopping" not in trainer_kwargs:
            trainer_kwargs["early_stopping"] = early_stopping

        # ----- 5. Create the runner and run training -----
        runner = TrainRunner(
            self,
            training_plan=training_plan,
            data_splitter=data_module,
            max_epochs=max_epochs,
            accelerator=accelerator,
            devices=devices,
            **trainer_kwargs,
        )
        runner()   # This calls Trainer.fit() and sets self.is_trained_ = True

# 3. Baseline Models Config and Training

In [ ]:
N_LATENT = 10
N_HIDDEN = 128
N_LAYERS = 1
BETA = 10
MAX_EPOCHS = 100
LR = 1e-3
CODE_RATE = 5 # r, default from dvae-ecc
N_CODE = N_LATENT * CODE_RATE
G = build_repetition_G(N_LATENT, CODE_RATE)
TRAIN_SIZE = 0.8
VAL_SIZE = 0.1

## SCVI

In [ ]:
CONT_MODEL_NAME = register_model("Baseline scVI") # Baseline model from scvi with continuous latent space
my_file = Path(model_path(CONT_MODEL_NAME))
if my_file.exists():
    cont_model = SCVI.load(my_file, adata=adata)
else:
    scvi.model.SCVI.setup_anndata(adata, layer="counts")
    test_model = scvi.model.SCVI(adata, n_latent=N_LATENT, n_hidden=N_HIDDEN, n_layers=N_LAYERS) # changing to 1 layer
    test_model.train(max_epochs=MAX_EPOCHS, train_size=TRAIN_SIZE, validation_size=VAL_SIZE, early_stopping=True, plan_kwargs={"lr": LR})
    # Save model
    test_model.save(my_file)
    cont_model = SCVI.load(my_file, adata=adata)

## Coded DSCVI

In [ ]:
CODED100_MODEL_NAME = register_model("Baseline DSCVI Coded") # The model we will be training
# Setup AnnData for the coded model (same data, no batch)
MODEL_NAME = CODED100_MODEL_NAME
my_file = Path(model_path(MODEL_NAME))
if my_file.exists():
    coded_model = DSCVI.load(my_file, adata=adata)  
else:
    DSCVI.setup_anndata(adata, layer="counts")
    coded_model = DSCVI(adata, n_latent=N_LATENT, n_code=N_CODE, G=G, coded_key=True,
                    n_hidden=N_HIDDEN, n_layers=N_LAYERS, beta=BETA, latent_distribution='ber')
    # Train with the same number of epochs
    coded_model.train(max_epochs=100, 
                      train_size=TRAIN_SIZE, 
                      validation_size=VAL_SIZE,
                      early_stopping=True,
                      plan_kwargs={"lr": LR})
    coded_model.save(my_file, overwrite=True) 
 

## Uncoded DSCVI

In [ ]:
UNCODED_MODEL_NAME = register_model("Baseline DSCVI Uncoded") # The model we will be training
# Setup AnnData for the coded model (same data, no batch)
MODEL_NAME = UNCODED_MODEL_NAME
my_file = Path(model_path(MODEL_NAME))
if my_file.exists():
    uncoded_model = DSCVI.load(my_file, adata=adata)  
else:
    DSCVI.setup_anndata(adata, layer="counts")
    uncoded_model = DSCVI(adata, n_latent=N_LATENT, coded_key=False,
                    n_hidden=N_HIDDEN, n_layers=N_LAYERS, beta=BETA, latent_distribution='ber')
    # Train with the same number of epochs
    uncoded_model.train(max_epochs=MAX_EPOCHS, 
                      train_size=TRAIN_SIZE, 
                      validation_size=VAL_SIZE,
                      early_stopping=True,
                      plan_kwargs={"lr": LR})
    uncoded_model.save(my_file, overwrite=True) 

## Gridsearch Model Learning Process

In [ ]:
from scvi.train import TrainingPlan
from pathlib import Path
import scvi

In [ ]:
from pathlib import Path


def _as_list(x):
    """Wrap a scalar as a one-element list; pass iterables through as a list."""
    if isinstance(x, (list, tuple, set)) or type(x).__name__ == "ndarray":
        return list(x)
    return [x]


def run_cyclical_gridsearch(
    models_dict,
    beta,
    ratio,
    cycles_grid=(2,),
    epochs_per_cycle_grid=(25, 30),
    kl_weight_history=None,
):
    """
    Train (or resume) a cyclical-KL gridsearch for DSCVI models.

    Parameters
    ----------
    models_dict : dict
        Dictionary to store trained/loaded models under `models_dict[name] = {"model": model}`.
        Modified in place.
    beta : float or iterable of float
        Beta value, or grid of beta values.
    ratio : float or iterable of float
        Ratio value, or grid of ratio values.
    cycles_grid : int or iterable of int, optional
        Number of cycles to grid over. Default (2,).
    epochs_per_cycle_grid : int or iterable of int, optional
        Epochs per cycle to grid over. Default (25, 30).
    kl_weight_history : list, optional
        If given, (model_name, epoch, kl_weight) tuples are appended per training epoch.
        If None, KL-weight tracking is skipped.
    """
    # Normalize scalars -> lists so the loops below are uniform
    beta_grid             = _as_list(beta)
    ratio_grid            = _as_list(ratio)
    cycles_grid           = _as_list(cycles_grid)
    epochs_per_cycle_grid = _as_list(epochs_per_cycle_grid)

    for cycles in cycles_grid:
        for epochs_per_cycle in epochs_per_cycle_grid:
            for ratio_val in ratio_grid:
                for beta_val in beta_grid:
                    max_epochs = cycles * epochs_per_cycle

                    model_key = (
                        f"Cyclical_{cycles}_DSCVI_epochs_{epochs_per_cycle}"
                        f"_ratio_{ratio_val}_beta_{beta_val}"
                    )
                    model_name = model_key.replace(".", "_").replace(" ", "_")

                    registered_name = register_model(model_name)
                    save_path = model_path(registered_name)
                    my_file = Path(save_path)

                    # ---------------- Load if it already exists ----------------
                    if my_file.exists():
                        print(f"\n{'='*50}")
                        print(f"Loading existing model: {model_name}")
                        print(f"Path: {my_file}")
                        print(f"{'='*50}\n")

                        DSCVI.setup_anndata(adata, layer="counts")
                        model = DSCVI.load(my_file, adata=adata)

                    # ---------------- Otherwise, train as before ----------------
                    else:
                        print(f"\n{'='*50}")
                        print(f"Training Model: {model_name}")
                        print(
                            f"Parameters: cycles={cycles}, beta={beta_val}, "
                            f"lr={LR}, ratio={ratio_val}"
                        )
                        print(f"{'='*50}\n")

                        # ---------- Custom Training Plan (captures current grid values) ----------
                        class CyclicalKLTrainingPlan(TrainingPlan):
                            def __init__(
                                self,
                                module,
                                n_epochs_per_cycle: int = epochs_per_cycle,
                                ratio: float = ratio_val,
                                min_kl_weight: float = 0.0,
                                max_kl_weight: float = 1.0,
                                **kwargs,
                            ):
                                self.n_epochs_per_cycle = n_epochs_per_cycle
                                self.ratio = ratio
                                self.min_kl_weight = min_kl_weight
                                self.max_kl_weight = max_kl_weight
                                self._kl_weight_value = 0.0
                                super().__init__(module, **kwargs)

                            @property
                            def kl_weight(self):
                                return self._kl_weight_value

                            @kl_weight.setter
                            def kl_weight(self, value):
                                self._kl_weight_value = value

                            def on_train_epoch_start(self) -> None:
                                """Update kl_weight at the start of every epoch."""
                                super().on_train_epoch_start()

                                if self.n_epochs_per_cycle <= 0:
                                    return

                                cycle_position = (
                                    self.current_epoch % self.n_epochs_per_cycle
                                ) / self.n_epochs_per_cycle

                                if cycle_position < self.ratio:
                                    self.kl_weight = self.min_kl_weight + (
                                        self.max_kl_weight - self.min_kl_weight
                                    ) * (cycle_position / self.ratio)
                                else:
                                    self.kl_weight = self.max_kl_weight

                                if kl_weight_history is not None:
                                    kl_weight_history.append(
                                        (model_name, self.current_epoch, self.kl_weight)
                                    )

                        # Setup AnnData
                        DSCVI.setup_anndata(adata, layer="counts")

                        # Instantiate model with this iteration's beta
                        model = DSCVI(
                            adata,
                            n_latent=N_LATENT,
                            n_code=N_CODE,
                            G=G,
                            coded_key=True,
                            n_hidden=N_HIDDEN,
                            n_layers=N_LAYERS,
                            beta=beta_val,
                            latent_distribution='ber',
                        )

                        model._training_plan_cls = CyclicalKLTrainingPlan

                        model.train(
                            max_epochs=max_epochs,
                            train_size=TRAIN_SIZE,
                            validation_size=VAL_SIZE,
                            early_stopping=True,
                            plan_kwargs={"lr": LR},   # lr kept constant
                        )

                        model.save(save_path, overwrite=True)

                    # ---------------- Store (loaded or freshly trained) ----------------
                    models_dict[model_name] = model


    return models_dict

In [ ]:
gridsearch_cyclical_kl_dict_5 = {}
kl_weight_history = []

run_cyclical_gridsearch(
    models_dict=gridsearch_cyclical_kl_dict_5,
    beta=[5.0, 10.0],       # grid
    ratio=0.5,              # scalar (broadcast)
    cycles_grid=2,
    epochs_per_cycle_grid=[25, 30],
    kl_weight_history=kl_weight_history,
)

In [ ]:
gridsearch_cyclical_kl_dict_10 = {}
kl_weight_history = []

run_cyclical_gridsearch(
    models_dict=gridsearch_cyclical_kl_dict_10,
    beta=10.0,            # scalar
    ratio=[0.4, 0.5, 0.6],              # scalar (broadcast)
    cycles_grid=[2, 3, 4, 5],
    epochs_per_cycle_grid=[35, 50, 65],
    kl_weight_history=kl_weight_history,
)

In [ ]:
gridsearch_cyclical_kl_dict_15 = {}
kl_weight_history_15 = []

run_cyclical_gridsearch(
    models_dict=gridsearch_cyclical_kl_dict_15,
    beta=15.0,            # scalar
    ratio=[0.4, 0.5, 0.6],              # scalar (broadcast)
    cycles_grid=[2, 3, 4, 5],
    epochs_per_cycle_grid=[35, 50, 65],
    kl_weight_history=kl_weight_history_15,
)

### Evaluate Results

In [ ]:
"""
Extended evaluation metrics for DSCVI / coded discrete VAE models.

Design notes / assumptions (adjust to match your actual model API):

- Codes are obtained via `get_representation(model, adata, indices, return_type=...)`,
  matching your existing helper. `code_type` below selects between:
    "qm" -> Q_M, the 50-bit error-correcting code (coded models)

- Full-pipeline reconstruction (`obs`, `imp`) is obtained via the same sampling logic
  as your `evaluate_posterior_predictive`: run `model._make_data_loader(...)`, then
  `model.module.sample(tensors, n_samples=...)`, averaging over the MC-sample axis
  (shape-agnostic to scVI's (n_samples, batch, genes) vs. DSCVI's (batch, genes,
  n_samples) convention). This covers masked reconstruction error and zero-fraction
  calibration, which only need whole-pipeline x -> x_hat.

- Bit-flip robustness is the one metric that needs a *standalone* decode-from-code
  step (encode x -> z, flip one bit -> decode flipped z -> compare), which isn't
  exposed by evaluate_posterior_predictive (it only decodes end-to-end from raw
  tensors). This code assumes `model.module.generative(z)` exists and returns
  something with a `.get("px_rate")`-style mean, or a tensor directly — CONFIRM
  this against your actual DSCVI module and adjust `_decode_from_code` if the
  method name / return signature differs. If no such method exists, bit-flip
  robustness isn't computable without adding one to the model.

- Masking is applied to `adata.X` (or `layer`, if you pass one) BEFORE it's handed
  to the model, since encoding/decoding now goes through `get_representation` /
  `_make_data_loader` on an AnnData rather than a raw array. Masking creates a
  copy of the AnnData subset — it does not mutate your original object.

- `cell_types` is read from `adata.obs["cell_type"]`, aligned to `indices`.
"""

import copy
import numpy as np
import torch
from scipy.stats import entropy as scipy_entropy
from scipy.spatial.distance import pdist


# ---------------------------------------------------------------------------
# Model I/O helpers — the three functions that touch model internals directly
# ---------------------------------------------------------------------------


def get_hard_codes(model, adata, indices=None, use_qm=True):
    """
    Extract hard-thresholded binary codes from the model.
    
    Parameters
    ----------
    model : DSCVI
        Trained discrete model.
    adata : AnnData
        Full AnnData object.
    indices : array-like, optional
        Subset of cells to evaluate.
    use_qm : bool
        If True, uses the 50-bit message distribution (Q_M).
        If False, uses the 10-bit latent distribution (Q_Z).
    
    Returns
    -------
    np.ndarray, shape (n_cells, bits)
        Binary codes (0 or 1) with no flattening.
    """
    if use_qm:
        # Get soft probabilities for the M-bit error-correcting code
        probs = model.get_latent_representation(
            adata, indices=indices, return_dist=False, give_mean=True
        )
    else:
        # Get soft probabilities for the D-bit latent (if needed)
        probs = model.get_latent_representation(
            adata, indices=indices, return_dist=False, give_mean=True, return_qz=True
        )
    
    # Threshold element-wise, preserving shape (n_cells, bits)
    if hasattr(probs, "cpu"):  # PyTorch tensor
        hard = (probs > 0.5).int().cpu().numpy()
    else:  # NumPy array
        hard = (probs > 0.5).astype(int)
    
    return hard


def get_representation(model, adata, indices=None, return_type="z"):
    """
    Your function, included here as-is except for the `results`/`resuls` typo fix
    and importing your real `get_hard_codes` instead of the stub above.
    """
    if return_type == "z":
        return model.get_latent_representation(
            adata=adata, indices=indices, return_dist=False, give_mean=False
        )
    elif return_type == "qm_soft":
        result = model.get_latent_representation(
            adata=adata, indices=indices, return_dist=False, give_mean=True
        )
    elif return_type == "qm_hard":
        result = get_hard_codes(model, adata=adata, indices=indices, use_qm=True)
    elif return_type == "qz_soft":
        result = model.get_latent_representation(
            adata=adata, indices=indices, return_dist=False, give_mean=True, return_qz=True
        )
    elif return_type == "qz_hard":
        result = get_hard_codes(model, adata=adata, indices=indices, use_qm=False)
    else:
        raise ValueError(f"Unknown return_type: {return_type}")

    return result


def _get_codes(model, adata, indices, code_type="qm"):
    """Fetch hard binary codes for a given code family ('qm' or 'qz')."""
    return_type = "qm_hard" if code_type == "qm" else "qz_hard"
    z = get_representation(model, adata, indices=indices, return_type=return_type)
    if torch.is_tensor(z):
        z = z.detach().cpu().numpy()
    return np.asarray(z).astype(np.int32)


def get_model_device(model):
    return next(model.module.parameters()).device


def _run_posterior_predictive(model, adata, indices=None, batch_size=128, n_samples=30):
    """
    Full-pipeline x -> x_hat via the same logic as evaluate_posterior_predictive.
    Returns (obs, imp) as numpy arrays, shape (n_cells, n_genes).
    """
    dataloader = model._make_data_loader(
        adata=adata, indices=indices, batch_size=batch_size, shuffle=False
    )

    all_observed, all_imputed = [], []
    device = get_model_device(model)

    with torch.inference_mode():
        for tensors in dataloader:
            tensors = {k: v.to(device) for k, v in tensors.items() if torch.is_tensor(v)}

            if "X" in tensors:
                x_obs = tensors["X"]
            elif "x" in tensors:
                x_obs = tensors["x"]
            else:
                raise KeyError(f"Available keys: {tensors.keys()}")

            samples = model.module.sample(tensors, n_samples=n_samples, max_poisson_rate=1e8)

            if n_samples > 1:
                if samples.shape[0] == n_samples:
                    x_imp = samples.mean(dim=0)   # scVI style
                else:
                    x_imp = samples.mean(dim=-1)  # DSCVI style
            else:
                x_imp = samples

            all_observed.append(x_obs.cpu().numpy())
            all_imputed.append(x_imp.cpu().numpy())

    obs = np.concatenate(all_observed, axis=0)
    imp = np.concatenate(all_imputed, axis=0)
    return obs, imp


def _decode_from_code(model, z, device=None):
    """
    Standalone code -> reconstruction, needed only for bit_flip_robustness.

    ASSUMPTION TO CONFIRM: model.module.generative(z) exists and returns either
    a tensor of reconstructed means, or a dict/namedtuple with a 'px_rate' (or
    similarly named) field. Adjust the unpacking below to match your DSCVI
    module's actual generative() signature and output.
    """
    device = device or get_model_device(model)
    model.module.eval()
    with torch.no_grad():
        z_t = torch.as_tensor(z, dtype=torch.float32, device=device)
        out = model.module.generative(z_t)
        if isinstance(out, dict):
            x_hat = out.get("px_rate", out.get("px_mean"))
        elif isinstance(out, (tuple, list)):
            x_hat = out[0]
        else:
            x_hat = out
        return x_hat.detach().cpu().numpy()


def _make_masked_adata_copy(adata, indices, mask_fraction, layer=None, seed=0):
    """
    Copy adata (subset to indices), zero out exactly `mask_fraction` of the 
    **nonzero entries** in .X (or `layer`), and return (masked_adata, mask, original_values) 
    so masked entries can be scored against ground truth after decoding.
    """
    rng = np.random.default_rng(seed)
    sub = adata[indices].copy() if indices is not None else adata.copy()

    mat = sub.layers[layer] if layer else sub.X
    mat = np.asarray(mat.todense()) if hasattr(mat, "todense") else np.asarray(mat)

    # 1. Extract all (cell, gene) indices where true count > 0
    nonzero_indices = np.argwhere(mat > 0)  # shape: (n_nonzero, 2)
    n_nonzero = len(nonzero_indices)

    if n_nonzero == 0:
        # Edge case: no nonzero entries to mask
        mask = np.zeros(mat.shape, dtype=bool)
        original_values = np.array([])
        masked_mat = mat.copy()
    else:
        n_mask = int(mask_fraction * n_nonzero)
        n_mask = max(1, min(n_mask, n_nonzero))  # ensure at least 1 and not exceed total

        shuffled_idx = rng.permutation(n_nonzero)
        selected_indices = nonzero_indices[shuffled_idx[:n_mask]]

        mask = np.zeros(mat.shape, dtype=bool)
        mask[selected_indices[:, 0], selected_indices[:, 1]] = True

        original_values = mat[mask].copy()

        masked_mat = mat.copy()
        masked_mat[mask] = 0.0

    if layer:
        sub.layers[layer] = masked_mat
    else:
        sub.X = masked_mat

    return sub, mask, original_values

import numpy as np
import torch
from scipy.special import gammaln
from scipy.sparse import issparse

def _get_px_distribution(model, adata, batch_size=128, layer=None, use_gpu=True):
    """
    Run a single forward pass (no sampling) on all cells in `adata`
    and return the `px` distribution object (e.g., ZeroInflatedNegativeBinomial).
    
    This is a batched implementation that returns the concatenated
    distribution parameters (mu, theta, pi) as numpy arrays.
    """

    device = model.device
    model.module.eval()
    
    # We'll collect per-gene parameter arrays
    mu_list, theta_list, pi_list = [], [], []
    
    n_cells = adata.n_obs
    n_vars = adata.n_vars
    
    # Get the data matrix
    mat = adata.X
    if issparse(mat):
        mat = mat.toarray()
    else:
        mat = np.asarray(mat)
    
    # Prepare batch indices (assuming adata.obs has a 'batch' column)
    # If not, you need to adjust. This is a typical scvi-tools setup.
    batch_ids = adata.obs['_batch_encoded'].values if '_batch_encoded' in adata.obs else np.zeros(n_cells, dtype=np.int64)
    # If you have other covariates, add them here.
    
    for start in range(0, n_cells, batch_size):
        end = min(start + batch_size, n_cells)
        batch_adata = adata[start:end]  # not used directly, we just need indices
        
        # Build tensors dictionary (adjust keys according to your model)
        # tensors = {
        #     REGISTRY_KEYS.BATCH_KEY: torch.tensor(batch_ids[start:end], device=device).unsqueeze(1),
        #     # Add other required keys if present (e.g., CONT_COVS_KEY, CAT_COVS_KEY, SIZE_FACTOR_KEY, LABELS_KEY)
        #     # For simplicity, we assume only batch is needed. Adjust accordingly.
        # }
        tensors = {
            REGISTRY_KEYS.BATCH_KEY: torch.tensor(batch_ids[start:end], device=device).unsqueeze(1),
            REGISTRY_KEYS.X_KEY: torch.tensor(mat[start:end], dtype=torch.float32, device=device),
        }

        if 'X' in tensors:
            x_obs = tensors['X']
        elif 'x' in tensors:
            x_obs = tensors['x']
        else:
            raise KeyError(f"Available keys: {tensors.keys()}")

        library_size = x_obs.sum(dim=-1, keepdim=True)

        if 'labels' in adata.obs:
            labels_batch = adata.obs['labels'].values[start:end]
            tensors[REGISTRY_KEYS.LABELS_KEY] = torch.tensor(labels_batch, dtype=torch.int64, device=device).unsqueeze(1)
        else:
            # Dummy labels (all zeros) – shape (batch_size, 1)
            tensors[REGISTRY_KEYS.LABELS_KEY] = torch.zeros((end - start, 1), dtype=torch.int64, device=device)

        # If you have size factors, add them:
        if hasattr(adata, 'obs') and 'size_factor' in adata.obs:
            size_factor = adata.obs['size_factor'].values[start:end]
            tensors[REGISTRY_KEYS.SIZE_FACTOR_KEY] = torch.tensor(size_factor, device=device).unsqueeze(1)
        
        # Forward pass (no sampling, n_samples=1)
        with torch.no_grad():
            _, generative_outputs = model.module.forward(
                tensors,
                inference_kwargs={"n_samples": 1},
                compute_loss=False,
            )
        
        dist = generative_outputs[MODULE_KEYS.PX_KEY]
        
        # Extract parameters (they should be on the same device)
        mu = dist.mu.cpu().numpy() if hasattr(dist, 'mu') else None
        theta = dist.theta.cpu().numpy() if hasattr(dist, 'theta') else None
        pi = torch.sigmoid(dist.zi_logits).cpu().numpy() if hasattr(dist, 'zi_logits') else None

        mu_scaled = mu * library_size.cpu().numpy()  # MISSING
        mu_list.append(mu_scaled)
        if theta is not None:
            theta_list.append(theta)
        if pi is not None:
            pi_list.append(pi)
    
    # Concatenate
    mu_all = np.concatenate(mu_list, axis=0) if mu_list else None
    theta_all = np.concatenate(theta_list, axis=0) if theta_list else None
    pi_all = np.concatenate(pi_list, axis=0) if pi_list else None
    
    return mu_all, theta_all, pi_all



# ---------------------------------------------------------------------------
# 1. Intracluster bit variance
# ---------------------------------------------------------------------------

def intracluster_bit_variance(codes, cell_types):
    """
    Mean per-bit variance within each cell-type cluster, averaged across clusters.
    Low variance = bits are stable/consistent within a biological cluster (good).
    High variance = bits are noisy or not capturing cluster identity.

    Returns: dict with 'mean_variance', 'per_bit_variance' (array), 'per_cluster' (dict)
    """
    codes = np.asarray(codes)
    cell_types = np.asarray(cell_types)
    unique_types = np.unique(cell_types)

    per_cluster = {}
    per_bit_vars = []

    for ct in unique_types:
        mask = cell_types == ct
        if mask.sum() < 2:
            continue
        cluster_codes = codes[mask]
        bit_var = cluster_codes.var(axis=0)  # variance per bit within this cluster
        per_cluster[str(ct)] = {
            "mean_bit_variance": float(bit_var.mean()),
            "per_bit_variance": bit_var,
        }
        per_bit_vars.append(bit_var)

    per_bit_vars = np.stack(per_bit_vars, axis=0)  # (n_clusters, n_bits)
    return {
        "mean_variance": float(per_bit_vars.mean()),
        "per_bit_variance": per_bit_vars.mean(axis=0),  # averaged across clusters
        "per_cluster": per_cluster,
    }


# ---------------------------------------------------------------------------
# 2. Shannon entropy (per-bit and joint codeword)
# ---------------------------------------------------------------------------

def shannon_entropy_bits(codes):
    """
    Per-bit entropy: catches individual collapsed bits (always 0 or always 1).
    Joint codeword entropy: catches codeword-level collapse that correlated bits
    can hide from per-bit entropy alone.

    Returns: dict with 'per_bit_entropy' (array, bits), 'mean_per_bit_entropy',
             'joint_entropy' (bits), 'max_joint_entropy' (bits, = n_bits),
             'joint_entropy_fraction'
    """
    codes = np.asarray(codes)
    n_cells, n_bits = codes.shape

    # per-bit entropy
    p1 = codes.mean(axis=0)  # P(bit=1) per bit
    per_bit_entropy = np.array([
        scipy_entropy([p, 1 - p], base=2) if 0 < p < 1 else 0.0
        for p in p1
    ])

    # joint entropy over observed codewords
    codeword_strs = ["".join(map(str, row)) for row in codes]
    unique, counts = np.unique(codeword_strs, return_counts=True)
    probs = counts / counts.sum()
    joint_entropy = scipy_entropy(probs, base=2)
    max_joint_entropy = n_bits  # theoretical max if all 2^n_bits codewords equally likely

    return {
        "per_bit_entropy": per_bit_entropy,
        "mean_per_bit_entropy": float(per_bit_entropy.mean()),
        "joint_entropy": float(joint_entropy),
        "max_joint_entropy": float(max_joint_entropy),
        "joint_entropy_fraction": float(joint_entropy / max_joint_entropy) if max_joint_entropy > 0 else 0.0,
    }


# ---------------------------------------------------------------------------
# 3. Intercluster code separation
# ---------------------------------------------------------------------------

def intercluster_code_separation(codes, cell_types, metric="hamming"):
    """
    Mean pairwise distance between cluster centroids, normalized by mean
    within-cluster spread. Uses Hamming distance by default since codes are
    discrete/binary (Euclidean distorts bit-space geometry).

    Returns: dict with 'separation_score' (higher = better separated),
             'mean_intercluster_distance', 'mean_intracluster_spread'
    """
    codes = np.asarray(codes, dtype=np.float64)
    cell_types = np.asarray(cell_types)
    unique_types = np.unique(cell_types)

    centroids = []
    intracluster_spreads = []

    for ct in unique_types:
        mask = cell_types == ct
        if mask.sum() < 2:
            continue
        cluster_codes = codes[mask]
        centroid = cluster_codes.mean(axis=0)
        centroids.append(centroid)
        # mean hamming distance of points to their own centroid (rounded), as spread proxy
        spread = np.abs(cluster_codes - centroid).mean()
        intracluster_spreads.append(spread)

    centroids = np.stack(centroids, axis=0)
    intercluster_dists = pdist(centroids, metric=metric)
    mean_inter = float(intercluster_dists.mean())
    mean_intra = float(np.mean(intracluster_spreads))

    separation_score = mean_inter / mean_intra if mean_intra > 0 else float("inf")

    return {
        "separation_score": separation_score,
        "mean_intercluster_distance": mean_inter,
        "mean_intracluster_spread": mean_intra,
    }


# ---------------------------------------------------------------------------
# 4. Codebook usage / coverage
# ---------------------------------------------------------------------------

def codebook_usage(codes, n_valid_codewords=None):
    """
    Fraction of the valid codeword space actually used, plus usage entropy.

    n_valid_codewords: for ECC-coded models, pass the number of valid codewords
    for that (n_bits, r) configuration — NOT 2**n_bits, since redundancy shrinks
    the valid space and coverage otherwise isn't comparable across r values.
    For uncoded models, leave as None (2**n_bits is usually too large to be a
    meaningful denominator; usage_entropy and n_unique_used are more informative
    in that case than a coverage fraction).

    Returns: dict with 'n_unique_used', 'coverage_fraction' (None if n_valid_codewords
             not given), 'usage_entropy', 'max_usage_entropy'
    """
    codes = np.asarray(codes)
    n_bits = codes.shape[1]
    codeword_strs = ["".join(map(str, row)) for row in codes]
    unique, counts = np.unique(codeword_strs, return_counts=True)
    n_unique_used = len(unique)

    probs = counts / counts.sum()
    usage_entropy = scipy_entropy(probs, base=2)
    max_usage_entropy = np.log2(n_valid_codewords) if n_valid_codewords else n_bits

    coverage_fraction = (n_unique_used / n_valid_codewords) if n_valid_codewords else None

    return {
        "n_unique_used": n_unique_used,
        "coverage_fraction": coverage_fraction,
        "usage_entropy": float(usage_entropy),
        "max_usage_entropy": float(max_usage_entropy),
        "usage_entropy_fraction": float(usage_entropy / max_usage_entropy) if max_usage_entropy > 0 else 0.0,
    }


# ---------------------------------------------------------------------------
# 5. Bit-flip robustness
# ---------------------------------------------------------------------------

def bit_flip_robustness(model, adata, indices, code_type="qm", n_cells_sample=500, seed=0):
    """
    For a sample of test cells: get hard codes -> flip each bit one at a
    time -> decode via model.module.generative() -> measure reconstruction
    degradation vs. the unflipped reconstruction. Averaged per bit and overall.

    A degenerate/collapsed code will show near-zero sensitivity to flips
    (every flip barely changes the reconstruction) — this doubles as a
    collapse detector distinct from your existing metrics.

    Requires `_decode_from_code` to correctly call your model's generative
    step from a code — see the assumption noted at the top of this file.

    Returns: dict with 'mean_degradation', 'per_bit_degradation' (array),
             'baseline_reconstruction_mse'
    """
    rng = np.random.default_rng(seed)
    indices = np.asarray(indices)
    sample_idx = rng.choice(indices, size=min(n_cells_sample, len(indices)), replace=False)

    x_true = adata[sample_idx].X
    x_true = np.asarray(x_true.todense()) if hasattr(x_true, "todense") else np.asarray(x_true)

    z = _get_codes(model, adata, sample_idx, code_type=code_type)
    n_bits = z.shape[1]

    x_hat_baseline = _decode_from_code(model, z)
    baseline_error = np.mean((x_true - x_hat_baseline) ** 2, axis=1)

    per_bit_degradation = np.zeros(n_bits)
    for b in range(n_bits):
        z_flipped = z.copy()
        z_flipped[:, b] = 1 - z_flipped[:, b]
        x_hat_flipped = _decode_from_code(model, z_flipped)
        flipped_error = np.mean((x_true - x_hat_flipped) ** 2, axis=1)
        per_bit_degradation[b] = float(np.mean(flipped_error - baseline_error))

    return {
        "mean_degradation": float(per_bit_degradation.mean()),
        "per_bit_degradation": per_bit_degradation,
        "baseline_reconstruction_mse": float(baseline_error.mean()),
    }


# ---------------------------------------------------------------------------
# 6. Masked reconstruction error using NLL and ZINB (input-masked, scored on masked entries only)
# ---------------------------------------------------------------------------
def masked_reconstruction_error(model, adata, indices, mask_fraction=0.1, layer=None,
                                batch_size=128, seed=0):
    # 1. Mask only nonzero entries
    masked_adata, mask, true_vals = _make_masked_adata_copy(
        adata, indices, mask_fraction, layer=layer, seed=seed
    )
    
    n_masked = mask.sum()
    if n_masked == 0:
        return {"masked_nll": np.nan, "masked_pearson": np.nan,
                "mask_fraction": mask_fraction, "n_masked_entries": 0}
    
    # 2. Full forward pass on ALL cells
    mu, theta, pi = _get_px_distribution(
        model, masked_adata, batch_size=batch_size, layer=layer
    )
    
    # 3. Slice to masked entries
    y = true_vals
    mu_masked = mu[mask]
    theta_masked = theta[mask] if theta is not None else None
    pi_masked = pi[mask] if pi is not None else None
    
    # 4. Compute NLL (with 1-pi correction)
    from scipy.special import gammaln
    
    if theta_masked is not None and pi_masked is not None:
        # NB log-likelihood
        nb_logL = (theta_masked * np.log(theta_masked + 1e-8)
                   + y * np.log(mu_masked + 1e-8)
                   - (y + theta_masked) * np.log(mu_masked + theta_masked + 1e-8)
                   + gammaln(y + theta_masked)
                   - gammaln(theta_masked)
                   - gammaln(y + 1))
        # Apply the 1-pi correction for positive counts
        logL = nb_logL + np.log(1 - pi_masked + 1e-8)   # <-- THIS IS THE LINE
    elif theta_masked is not None:
        # NB only
        logL = (theta_masked * np.log(theta_masked + 1e-8)
                + y * np.log(mu_masked + 1e-8)
                - (y + theta_masked) * np.log(mu_masked + theta_masked + 1e-8)
                + gammaln(y + theta_masked)
                - gammaln(theta_masked)
                - gammaln(y + 1))
    else:
        # Poisson fallback
        logL = y * np.log(mu_masked + 1e-8) - mu_masked - gammaln(y + 1)
    
    nll = -np.mean(logL)
    nll_median = -np.median(logL)
    
    # Optional Pearson (for sanity)
    if y.std() > 0 and mu_masked.std() > 0:
        pearson_r = np.corrcoef(y, mu_masked)[0, 1]
    else:
        pearson_r = np.nan
    
    return {
        "masked_nll": nll,
        "masked_nll_median": nll_median,
        "masked_pearson": pearson_r,
        "mask_fraction": mask_fraction,
        "n_masked_entries": int(n_masked),
    }


# ---------------------------------------------------------------------------
# 7. Zero-fraction calibration error
# ---------------------------------------------------------------------------

def zero_fraction_calibration_error(model, adata, indices, batch_size=128, n_samples=30,
                                     zero_threshold=1e-3):
    """
    Compares the fraction of zero entries per gene in true test data vs.
    reconstructed data (obtained via the full posterior-predictive pipeline,
    same as evaluate_posterior_predictive). For count-based single-cell data,
    a model that gets mean expression right but miscalibrates sparsity
    (e.g. under- or over-predicting dropout) will look fine on Pearson/MSE
    while being wrong in a way that matters biologically.

    `zero_threshold` must match your decoder's output scale — check a
    histogram of x_hat values if unsure; 1e-3 assumes rate-scale output,
    not raw counts.

    Returns: dict with 'mean_abs_calibration_error', 'per_gene_error' (array),
             'true_zero_fraction' (array), 'pred_zero_fraction' (array)
    """
    obs, x_hat = _run_posterior_predictive(
        model, adata, indices=indices, batch_size=batch_size, n_samples=n_samples
    )

    true_zero_frac = (obs == 0).mean(axis=0)
    pred_zero_frac = (x_hat < zero_threshold).mean(axis=0)
    per_gene_error = np.abs(true_zero_frac - pred_zero_frac)

    return {
        "mean_abs_calibration_error": float(per_gene_error.mean()),
        "per_gene_error": per_gene_error,
        "true_zero_fraction": true_zero_frac,
        "pred_zero_fraction": pred_zero_frac,
    }

import numpy as np
from scipy.stats import nbinom

def zero_fraction_calibration_error(model, adata, indices, batch_size=128, n_samples=30,
                                     zero_threshold=1e-3, layer=None):
    """
    Compares the fraction of zero entries per gene in true test data vs.
    reconstructed data sampled from the model's ZINB distribution (obtained
    via _get_px_distribution). This directly measures calibration of sparsity
    without relying on the posterior predictive pipeline.

    Parameters
    ----------
    model : scvi.nn.VAE or similar
        Trained model with decoder that outputs ZINB parameters.
    adata : AnnData
        Full dataset (used to pass through the model).
    indices : array-like
        Indices of cells to evaluate (typically test set).
    batch_size : int
        Mini-batch size for passing data through the model.
    n_samples : int
        Number of Monte Carlo samples from the posterior to average over.
    zero_threshold : float
        Threshold below which a reconstructed value is considered zero.
        This must match the scale of the decoder output (e.g., rate-scale).
    layer : str, optional
        Which layer of adata to use (e.g., 'counts').

    Returns
    -------
    dict with:
        'mean_abs_calibration_error' : float
        'per_gene_error' : np.ndarray (G,)
        'true_zero_fraction' : np.ndarray (G,)
        'pred_zero_fraction' : np.ndarray (G,)
    """
    # 1. Get the ZINB parameters for the requested cells
    mu, theta, pi = _get_px_distribution(
        model, adata, batch_size=batch_size, layer=layer
    )
    # mu, theta, pi are each of shape (n_cells, n_genes)
    # Select only the test indices
    mu = mu[indices] if isinstance(mu, np.ndarray) else mu[indices]
    theta = theta[indices] if isinstance(theta, np.ndarray) else theta[indices]
    pi = pi[indices] if isinstance(pi, np.ndarray) else pi[indices]

    n_cells, n_genes = mu.shape

    # 2. Sample from the ZINB distribution for each cell and gene
    # We will average over n_samples to get a robust estimate of the zero fraction.
    # Initialize an array to accumulate reconstructed counts across samples
    sampled_counts = np.zeros((n_cells, n_genes))

    for _ in range(n_samples):
        # Draw zero-inflation masks: 1 if zero-inflated (i.e., forced to zero)
        zi_mask = np.random.binomial(1, pi)  # shape (n_cells, n_genes)

        # For non-zero-inflated entries, sample from Negative Binomial
        # NB parameterization in scVI: mean = mu, dispersion = theta.
        # scipy's nbinom uses r = theta (dispersion) and p = theta / (theta + mu)
        # for mean = r*(1-p)/p = mu.
        p_nb = theta / (theta + mu)  # shape (n_cells, n_genes)
        # Ensure numerical stability: clip p to [eps, 1-eps]
        eps = 1e-7
        p_nb = np.clip(p_nb, eps, 1 - eps)
        # Sample from Negative Binomial
        nb_samples = nbinom.rvs(n=theta, p=p_nb, size=(n_cells, n_genes))
        # Apply zero inflation: if zi_mask == 1, set to 0, else keep nb_samples
        sample = np.where(zi_mask == 1, 0, nb_samples)

        sampled_counts += sample

    # Average over samples to get expected reconstructed counts
    reconstructed = sampled_counts / n_samples

    # 3. Compute zero fractions
    true_zero_frac = (adata[indices].X == 0).mean(axis=0)
    # If adata.X is a sparse matrix, convert to dense or use .A
    # For simplicity, we assume dense; otherwise adjust.

    pred_zero_frac = (reconstructed < zero_threshold).mean(axis=0)

    per_gene_error = np.abs(true_zero_frac - pred_zero_frac)

    return {
        "mean_abs_calibration_error": float(per_gene_error.mean()),
        "per_gene_error": per_gene_error,
        "true_zero_fraction": true_zero_frac,
        "pred_zero_fraction": pred_zero_frac,
    }

# ---------------------------------------------------------------------------
# 8. Reconstruction error and ELBO
# ---------------------------------------------------------------------------

import numpy as np
import pandas as pd
import torch

import numpy as np
import torch

import numpy as np
import pandas as pd
import torch

def extract_metrics(model, adata, name=None):
    """
    Extract reconstruction error and ELBO on train and test sets.
    - Train metrics: robustly read from model.history (supports kl_local_train).
    - Test metrics: uses model.get_elbo(adata, indices) and
      model.get_reconstruction_error(adata, indices)['reconstruction_loss'].
    Returns a dictionary with scalar floats.
    """
    # --- Helper: convert tensor/array to scalar (mean if multi-element) ---
    def to_scalar(x):
        if x is None:
            return np.nan
        if isinstance(x, (pd.Series, pd.DataFrame)):
            if len(x) == 1:
                x = x.iloc[0]
            else:
                x = x.mean()
        if isinstance(x, (np.ndarray, torch.Tensor)):
            if x.numel() == 1:
                return float(x.item() if hasattr(x, 'item') else x)
            else:
                return float(x.mean().item() if hasattr(x, 'mean') else np.mean(x))
        try:
            return float(x)
        except (TypeError, ValueError):
            return np.nan

    # ------------------------------------------------------------------
    # 1. TRAINING METRICS: robustly parse model.history
    # ------------------------------------------------------------------
    hist = model.history
    if isinstance(hist, pd.DataFrame):
        df_hist = hist
    elif isinstance(hist, dict):
        try:
            df_hist = pd.DataFrame(hist)
        except ValueError:
            df_hist = pd.DataFrame([hist])
    elif isinstance(hist, list) and hist and isinstance(hist[0], dict):
        df_hist = pd.DataFrame(hist)
    else:
        df_hist = pd.DataFrame(hist)

    # --- Reconstruction loss ---
    recon_col = 'reconstruction_loss_train'
    if recon_col is None:
        # Fallback: any column containing 'reconstruction' and 'train'
        alt = [c for c in df_hist.columns if 'reconstruction' in c and 'train' in c]
        if alt:
            recon_col = alt[0]
    train_recon = to_scalar(df_hist[recon_col].iloc[-1]) if recon_col else np.nan

    # --- ELBO (which is 'kl_local_train' in your history) ---
    elbo_col = 'elbo_train'
    if elbo_col is None:
        # Fallback: any column containing 'kl' and 'train' OR 'elbo'
        alt = [c for c in df_hist.columns if ('kl' in c and 'train' in c) or 'elbo' in c]
        if alt:
            elbo_col = alt[0]
    train_elbo = to_scalar(df_hist[elbo_col].iloc[-1]) if elbo_col else np.nan

    # ------------------------------------------------------------------
    # 2. TEST METRICS: use model.get_* methods with indices
    # ------------------------------------------------------------------
    test_indices = getattr(model, "test_indices", None)
    if test_indices is None or len(test_indices) == 0:
        test_indices = getattr(model, "validation_indices", None)
        if test_indices is None or len(test_indices) == 0:
            return {
                'reconstruction_train': train_recon,
                'reconstruction_test': np.nan,
                'elbo_train': train_elbo,
                'elbo_test': np.nan,
            }

    # Helper to call method and convert result to scalar
    def get_metric(method_name, indices):
        method = getattr(model, method_name, None)
        if method is None:
            return np.nan
        try:
            # Try (adata, indices) first
            result = method(adata, indices)
        except TypeError:
            # Fallback to (indices=..., return_mean=True)
            result = method(indices=indices, return_mean=True)
        # If result is a dict, extract the 'reconstruction_loss' key
        if isinstance(result, dict) and 'reconstruction_loss' in result:
            result = result['reconstruction_loss']
        return to_scalar(result)

    test_recon = get_metric('get_reconstruction_error', test_indices)
    test_elbo = get_metric('get_elbo', test_indices)

    return {
        'reconstruction_train': train_recon,
        'reconstruction_test': test_recon,
        'elbo_train': train_elbo,
        'elbo_test': -test_elbo,
    }

# ---------------------------------------------------------------------------
# Harness — loops over your {model_name: model} dict
# ---------------------------------------------------------------------------

def compute_extended_metrics(
    models,
    adata,
    code_types=None,
    n_valid_codewords=None,
    mask_fraction=0.1,
    bit_flip_sample_size=500,
    n_samples=30,
    batch_size=128,
    layer=None,
):
    """
    models: dict {model_name: model_object}
    adata: full AnnData; train_indices / test_indices index into it
    code_types: dict {model_name: "qm" | "qz"} — which code family to evaluate
                per model (use "qm" for coded/ECC models, "qz" for uncoded).
                Defaults to "qm" for every model if not specified.
    n_valid_codewords: dict {model_name: int} for coded models, or None to skip
                        coverage_fraction (usage_entropy is still computed).
    layer: adata layer holding raw counts, if not adata.X.

    Returns: nested dict {model_name: {metric_name: value_dict}}
    """
    code_types = code_types or {}
    n_valid_codewords = n_valid_codewords or {}

    results = {}

    for name, model in models.items():
        ctype = code_types.get(name, "qm")
        model = models[name]
        train_indices = model.train_indices
        test_indices = model.test_indices
        cell_types_train = np.asarray(adata.obs["cell_type"])[train_indices]
        cell_types_test = np.asarray(adata.obs["cell_type"])[test_indices]
        z_train = _get_codes(model, adata, train_indices, code_type=ctype)
        z_test = _get_codes(model, adata, test_indices, code_type=ctype)

        model_results = {
            "intracluster_bit_variance_train": intracluster_bit_variance(z_train, cell_types_train),
            "intracluster_bit_variance_test": intracluster_bit_variance(z_test, cell_types_test),
            "shannon_entropy_train": shannon_entropy_bits(z_train),
            "shannon_entropy_test": shannon_entropy_bits(z_test),
            "intercluster_separation_test": intercluster_code_separation(z_test, cell_types_test),
            "codebook_usage_train": codebook_usage(z_train, n_valid_codewords.get(name)),
            "codebook_usage_test": codebook_usage(z_test, n_valid_codewords.get(name)),
            # "bit_flip_robustness_test": bit_flip_robustness(
            #   model, adata, test_indices, code_type=ctype, n_cells_sample=bit_flip_sample_size
            # ),
            "masked_reconstruction_error_test": masked_reconstruction_error(
                model, adata, test_indices, mask_fraction=mask_fraction,
                layer=layer, batch_size=batch_size
            ),
            "zero_fraction_calibration_error_test": zero_fraction_calibration_error(
                model, adata, test_indices, batch_size=batch_size, n_samples=n_samples
            ),
            "reconstruction_error_elbo": extract_metrics(model, adata)
        }
        results[name] = model_results

    return results


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np


# Default output location, relative to the repo root
DEFAULT_TABLES_DIR = Path("scvi_data") / "tables"
DEFAULT_CSV_NAME   = "extended_metrics_test.csv"


def metrics_to_dataframe(
    results,
    save_csv=False,
    csv_path=None,
    csv_name=DEFAULT_CSV_NAME,
    decimals=None,
):
    """
    Flatten the nested dict from `compute_extended_metrics` into a tidy DataFrame
    with one row per model and the requested test-set metrics as columns.

    Columns (all evaluated on the TEST set):
        Model : model name (row identifier)
        MIE   : masked reconstruction error (mean NLL on masked entries)  [lower = better]
        MIEM  : masked reconstruction error, median NLL                   [lower = better]
        MP    : masked Pearson correlation                                [higher = better]
        RE    : reconstruction error (test)                               [lower = better]
        ELBO  : ELBO (test; already sign-flipped so higher = better)
        ZFCE  : zero-fraction calibration error (mean abs per-gene error) [lower = better]
        IBV   : intracluster bit variance (mean over clusters)            [lower = better]
        ISS   : intercluster separation score                             [higher = better]
        ENT   : joint codeword entropy (bits)                             [higher = better]
        COV   : codebook coverage as a percentage of valid codewords      [higher = better]
                (NaN if `n_valid_codewords` wasn't supplied for that model)

    Parameters
    ----------
    results : dict
        Output of compute_extended_metrics: {model_name: {metric_name: value_dict}}.
    save_csv : bool, optional
        If True, write the DataFrame to a CSV. Default False.
    csv_path : str or Path, optional
        Explicit destination for the CSV. If None, the file is written to
        `scvi_data/tables/<csv_name>` (directory created if missing).
    csv_name : str, optional
        Filename used when `csv_path` is None. Default "extended_metrics_test.csv".
    decimals : int or None, optional
        If given, round all numeric metric columns to this many decimals.

    Returns
    -------
    pd.DataFrame
    """
    rows = []
    for name, m in results.items():
        masked     = m.get("masked_reconstruction_error_test", {}) or {}
        recon_elbo = m.get("reconstruction_error_elbo", {}) or {}
        zfce       = m.get("zero_fraction_calibration_error_test", {}) or {}
        ibv        = m.get("intracluster_bit_variance_test", {}) or {}
        iss        = m.get("intercluster_separation_test", {}) or {}
        ent        = m.get("shannon_entropy_test", {}) or {}
        cov        = m.get("codebook_usage_test", {}) or {}

        cov_frac = cov.get("coverage_fraction", None)
        cov_pct  = float(cov_frac) * 100.0 if cov_frac is not None else np.nan

        rows.append({
            "Model": name,
            "MIE":   masked.get("masked_nll", np.nan),
            "MIEM":  masked.get("masked_nll_median", np.nan),
            "MP":    masked.get("masked_pearson", np.nan),
            "RE":    recon_elbo.get("reconstruction_test", np.nan),
            "ELBO":  recon_elbo.get("elbo_test", np.nan),
            "ZFCE":  zfce.get("mean_abs_calibration_error", np.nan),
            "IBV":   ibv.get("mean_variance", np.nan),
            "ISS":   iss.get("separation_score", np.nan),
            "ENT":   ent.get("joint_entropy", np.nan),
            "COV":   cov_pct,
        })

    df = pd.DataFrame(rows)

    # Force every metric column to float64 so rounding is reliable
    metric_cols = ["MIE", "MIEM", "MP", "RE", "ELBO", "ZFCE", "IBV", "ISS", "ENT", "COV"]
    for col in metric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype(float)

    if decimals is not None:
        df[metric_cols] = df[metric_cols].round(decimals)

    if save_csv:
        # Resolve destination: explicit path wins, otherwise default repo location
        if csv_path is None:
            dest = DEFAULT_TABLES_DIR / csv_name
        else:
            dest = Path(csv_path)

        # Create parent directory (scvi_data/tables/) if missing
        dest.parent.mkdir(parents=True, exist_ok=True)

        df.to_csv(dest, index=False)
        print(f"✅ Saved metrics table to {dest.resolve()}")

    return df

In [ ]:
DEFAULT_TABLES_DIR = Path("scvi_data") / "tables"
DEFAULT_CSV_NAME   = "model_metrics_test.csv"

In [ ]:
# One valid-codeword mapping per gridsearch dictionary, kept separate so a
# later assignment can't silently overwrite an earlier one.
valid_codeword_5 = {key: 1024 for key in gridsearch_cyclical_kl_dict_5}
valid_codeword_10 = {key: 1024 for key in gridsearch_cyclical_kl_dict_10}
valid_codeword_15 = {key: 1024 for key in gridsearch_cyclical_kl_dict_15}


In [ ]:
results1 = compute_extended_metrics(
    models=gridsearch_cyclical_kl_dict_15,
    adata=adata,
    n_valid_codewords=valid_codeword_15
)
results1


In [ ]:
df_metrics = metrics_to_dataframe(
    results1,
    save_csv=True,
    csv_name="beta15_test_metrics.csv",
    decimals=3,
)
df_metrics

In [ ]:
results2 = compute_extended_metrics(
    models=gridsearch_cyclical_kl_dict_10,
    adata=adata,
    n_valid_codewords=valid_codeword_10
)
results2


In [ ]:
df_metrics = metrics_to_dataframe(
    results2,
    save_csv=True,
    csv_name="beta10_test_metrics.csv",
    decimals=3,
)
df_metrics

In [ ]:
results3 = compute_extended_metrics(
    models=gridsearch_cyclical_kl_dict_5,
    adata=adata,
    n_valid_codewords=valid_codeword_5
)
results3


In [ ]:
df_metrics = metrics_to_dataframe(
    results3,
    save_csv=True,
    csv_name="beta5_test_metrics.csv",
    decimals=3,
)
df_metrics

## Gridsearch Model Coded ECC

In [ ]:
from pathlib import Path


def _as_list(x):
    """Wrap a scalar as a one-element list; pass iterables through as a list."""
    if isinstance(x, (list, tuple, set)) or type(x).__name__ == "ndarray":
        return list(x)
    return [x]


def run_ecc_gridsearch(
    models_dict,
    beta,
    ratio,
    n_latent,
    code_rate,
    cycles_grid=(2,),
    epochs_per_cycle_grid=(25, 30),
    kl_weight_history=None,
):
    """
    Train (or resume) a cyclical-KL gridsearch for DSCVI models.

    Returns
    -------
    models_dict : dict
        Updated dict of trained/loaded models.
    valid_codewords : dict
        Mapping {model_name: n_latent**2} for every model produced by this gridsearch.
    """
    # Normalize scalars -> lists so the loops below are uniform
    beta_grid             = _as_list(beta)
    ratio_grid            = _as_list(ratio)
    n_latent_grid         = _as_list(n_latent)
    code_rate_grid        = _as_list(code_rate)
    cycles_grid           = _as_list(cycles_grid)
    epochs_per_cycle_grid = _as_list(epochs_per_cycle_grid)

    # One valid-codeword mapping per gridsearch, kept separate from models_dict
    valid_codewords = {}

    for cycles in cycles_grid:
        for epochs_per_cycle in epochs_per_cycle_grid:
            for ratio_val in ratio_grid:
                for beta_val in beta_grid:
                    for n_latent_val in n_latent_grid:
                        for code_rate_val in code_rate_grid:
                            # Derive n_code and build G from the code rate
                            n_code_val = code_rate_val * n_latent_val
                            G = build_repetition_G(n_latent_val, code_rate_val)

                            max_epochs = cycles * epochs_per_cycle

                            model_key = (
                                f"Cyclical_{cycles}_DSCVI_epochs_{epochs_per_cycle}"
                                f"_ratio_{ratio_val}_beta_{beta_val}"
                                f"_nlatent_{n_latent_val}_coderate_{code_rate_val}"
                            )
                            model_name = model_key.replace(".", "_").replace(" ", "_")

                            # Record the number of valid codewords for this model
                            valid_codewords[model_name] = n_latent_val ** 2

                            registered_name = register_model(model_name)
                            save_path = model_path(registered_name)
                            my_file = Path(save_path)

                            # ---------------- Load if it already exists ----------------
                            if my_file.exists():
                                print(f"\n{'='*50}")
                                print(f"Loading existing model: {model_name}")
                                print(f"Path: {my_file}")
                                print(f"{'='*50}\n")

                                DSCVI.setup_anndata(adata, layer="counts")
                                model = DSCVI.load(my_file, adata=adata)

                            # ---------------- Otherwise, train as before ----------------
                            else:
                                print(f"\n{'='*50}")
                                print(f"Training Model: {model_name}")
                                print(
                                    f"Parameters: cycles={cycles}, beta={beta_val}, "
                                    f"lr={LR}, ratio={ratio_val}, "
                                    f"n_latent={n_latent_val}, code_rate={code_rate_val}, "
                                    f"n_code={n_code_val}"
                                )
                                print(f"{'='*50}\n")

                                # ---------- Custom Training Plan (captures current grid values) ----------
                                class CyclicalKLTrainingPlan(TrainingPlan):
                                    def __init__(
                                        self,
                                        module,
                                        n_epochs_per_cycle: int = epochs_per_cycle,
                                        ratio: float = ratio_val,
                                        min_kl_weight: float = 0.0,
                                        max_kl_weight: float = 1.0,
                                        **kwargs,
                                    ):
                                        self.n_epochs_per_cycle = n_epochs_per_cycle
                                        self.ratio = ratio
                                        self.min_kl_weight = min_kl_weight
                                        self.max_kl_weight = max_kl_weight
                                        self._kl_weight_value = 0.0
                                        super().__init__(module, **kwargs)

                                    @property
                                    def kl_weight(self):
                                        return self._kl_weight_value

                                    @kl_weight.setter
                                    def kl_weight(self, value):
                                        self._kl_weight_value = value

                                    def on_train_epoch_start(self) -> None:
                                        """Update kl_weight at the start of every epoch."""
                                        super().on_train_epoch_start()

                                        if self.n_epochs_per_cycle <= 0:
                                            return

                                        cycle_position = (
                                            self.current_epoch % self.n_epochs_per_cycle
                                        ) / self.n_epochs_per_cycle

                                        if cycle_position < self.ratio:
                                            self.kl_weight = self.min_kl_weight + (
                                                self.max_kl_weight - self.min_kl_weight
                                            ) * (cycle_position / self.ratio)
                                        else:
                                            self.kl_weight = self.max_kl_weight

                                        if kl_weight_history is not None:
                                            kl_weight_history.append(
                                                (model_name, self.current_epoch, self.kl_weight)
                                            )

                                # Setup AnnData
                                DSCVI.setup_anndata(adata, layer="counts")

                                # Instantiate model with this iteration's beta, n_latent, n_code, G
                                model = DSCVI(
                                    adata,
                                    n_latent=n_latent_val,
                                    n_code=n_code_val,
                                    G=G,
                                    coded_key=True,
                                    n_hidden=N_HIDDEN,
                                    n_layers=N_LAYERS,
                                    beta=beta_val,
                                    latent_distribution='ber',
                                )

                                model._training_plan_cls = CyclicalKLTrainingPlan

                                model.train(
                                    max_epochs=max_epochs,
                                    train_size=TRAIN_SIZE,
                                    validation_size=VAL_SIZE,
                                    early_stopping=True,
                                    plan_kwargs={"lr": LR},   # lr kept constant
                                )

                                model.save(save_path, overwrite=True)

                            # ---------------- Store (loaded or freshly trained) ----------------
                            models_dict[model_name] = model

    return models_dict, valid_codewords

In [ ]:
gridsearch_latent_dict = {}
kl_weight_history_latent = []

_, valid_codewords = run_ecc_gridsearch(
    models_dict=gridsearch_latent_dict,
    beta=10.0,       # grid
    ratio=0.5,              # scalar (broadcast)
    n_latent=[8, 9, 10, 11, 12],
    code_rate=[4, 5, 6],
    cycles_grid=2,
    epochs_per_cycle_grid=35,
    kl_weight_history=kl_weight_history_latent,
)

### Evaluate Results

In [ ]:
results4 = compute_extended_metrics(
    models=gridsearch_latent_dict,
    adata=adata,
    n_valid_codewords=valid_codewords
)
df_metrics = metrics_to_dataframe(
    results4,
    save_csv=True,
    csv_name="latent_ecc_test_metrics.csv",
    decimals=3,
)
df_metrics